# 衔接BND_annotate.ipynb，对添加注释后的结果进行统计并绘图。

## 一、提取需要的数据

### build_sv_upset_events_from_bnd_annotated_vcfs.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import gzip
import re
from collections import defaultdict
from pathlib import Path


TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]
BND_RE = re.compile(r"[\[\]]([^:\[\]]+):([0-9]+)[\[\]]")
GENERIC_TOKENS = {
    "", ".", "vcf", "vcfs", "result", "results", "output", "outputs", "variant", "variants",
    "work", "tmp", "temp", "filter", "filtered", "somatic", "somaticsv", "diploidsv",
    "candidatesv", "candidate", "final", "pon", "pon_filtered", "bnd_reclass",
    "cue", "lumpy", "gridss", "gripss", "manta", "delly", "svaba",
    "cue_result", "lumpy_result", "gridss_result", "manta_result", "delly_result", "svaba_result",
}


def open_text(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")


def parse_info(info):
    d = {}
    if info in {"", "."}:
        return d
    for item in info.split(";"):
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            d[k] = v
        else:
            d[item] = True
    return d


def chrom_norm(chrom, keep_chr=True):
    chrom = str(chrom or "")
    if keep_chr:
        return chrom if chrom.lower().startswith("chr") else "chr" + chrom
    return chrom[3:] if chrom.lower().startswith("chr") else chrom


def chrom_key(chrom):
    c = chrom_norm(chrom, keep_chr=False)
    if c.isdigit():
        return (0, int(c))
    rank = {"X": 23, "Y": 24, "M": 25, "MT": 25}.get(c.upper())
    if rank is not None:
        return (0, rank)
    return (1, c)


def to_int(x, default=None):
    try:
        if x in {None, "", ".", "NA"}:
            return default
        return int(float(str(x).split(",")[0]))
    except Exception:
        return default


def norm_svtype(sv):
    sv = str(sv or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DELETION"}:
        return "DEL"
    if sv in {"DUPLICATION"}:
        return "DUP"
    if sv in {"INVERSION"}:
        return "INV"
    if sv in {"INSERTION"}:
        return "INS"
    if sv in {"TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BREAKEND"}:
        return "BND"
    return sv



def extract_svtype(info_text, alt, info=None):
    if info is None:
        info = {}

    for key in ("SVTYPE", "svtype", "SvType"):
        if key in info:
            sv = norm_svtype(info.get(key))
            if sv != "UNKNOWN":
                return sv

    m = re.search(r'(?i)(?:^|;)SVTYPE=([^;\t\r\n ]+)', info_text or "")
    if m:
        sv = norm_svtype(m.group(1))
        if sv != "UNKNOWN":
            return sv

    alt_text = str(alt or "").strip().upper()
    m = re.search(r'<([^<>:,]+)>', alt_text)
    if m:
        sv = norm_svtype(m.group(1))
        if sv != "UNKNOWN":
            return sv

    if "[" in alt_text or "]" in alt_text:
        return "BND"

    return "UNKNOWN"

def infer_tool(path):
    s = str(path).lower()
    for tool in TOOLS:
        if tool in s:
            return tool
    if "gripss" in s:
        return "gridss"
    return "unknown"


def strip_extensions(name):
    for suffix in [".vcf.gz", ".vcf"]:
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name


def clean_sample_token(name):
    x = strip_extensions(Path(name).name)
    patterns = [
        r"\.bnd_reclass$",
        r"_somatic$",
        r"\.somatic\.sv$",
        r"\.somatic\.indel$",
        r"\.somatic\.filtered$",
        r"\.somatic$",
        r"\.gripss\.pon_filtered$",
        r"\.gripss\.filtered$",
        r"\.gridss\.filtered$",
        r"\.gridss$",
        r"\.delly$",
        r"\.lumpy$",
        r"\.manta$",
        r"\.svaba$",
        r"_raw$",
    ]
    changed = True
    while changed:
        changed = False
        for pat in patterns:
            y = re.sub(pat, "", x, flags=re.IGNORECASE)
            if y != x:
                x = y
                changed = True
    for tool in TOOLS + ["gripss"]:
        x = re.sub(rf"(^|[._-]){tool}([._-]|$)", r"\1", x, flags=re.IGNORECASE)
    x = re.sub(r"[._-]+$", "", x)
    return x


def looks_like_sample_token(token):
    t = str(token or "").strip()
    if not t:
        return False
    if t.lower() in GENERIC_TOKENS:
        return False
    if not re.fullmatch(r"[A-Za-z]*[0-9]{3,}[A-Za-z]?", t):
        return False
    return True


def looks_like_pair_token(token):
    t = str(token or "").strip()
    if "_vs_" not in t:
        return False
    a, b = t.split("_vs_", 1)
    return looks_like_sample_token(a) and looks_like_sample_token(b)


def candidate_tokens(path):
    p = Path(path)
    raw = [p.name]
    for parent in list(p.parents)[:6]:
        raw.append(parent.name)

    tokens = []
    for item in raw:
        token = clean_sample_token(item)
        pair_matches = re.findall(
            r"([A-Za-z]*[0-9]{3,}[A-Za-z]?_vs_[A-Za-z]*[0-9]{3,}[A-Za-z]?)",
            token,
        )
        for pair_token in pair_matches:
            if looks_like_pair_token(pair_token):
                tokens.append(pair_token)
        if looks_like_sample_token(token):
            tokens.append(token)
            patient = patient_from_sample(token)
            if looks_like_sample_token(patient):
                tokens.append(patient)

    seen = set()
    out = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            out.append(token)
    return out


def patient_from_sample(sample):
    if sample.endswith(("T", "N", "P")) and len(sample) > 1:
        return sample[:-1]
    return sample


def read_pair_list(path):
    pairs = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            tumor = row.get("tumor_id") or row.get("tumor") or row.get("TUMOR") or row.get("sample_tumor")
            normal = row.get("normal_id") or row.get("normal") or row.get("NORMAL") or row.get("sample_normal")
            pair_id = row.get("pair_id") or row.get("pair")
            if not tumor or not normal:
                continue
            if not pair_id:
                pair_id = f"{tumor}_vs_{normal}"
            pairs.append({"pair_id": pair_id, "tumor_id": tumor, "normal_id": normal, "patient": patient_from_sample(tumor)})
    return pairs


def discover_pairs(files_by_tool):
    samples = set()
    patients_from_pair_files = set()
    for tool, paths in files_by_tool.items():
        for p in paths:
            for token in candidate_tokens(p):
                if looks_like_pair_token(token):
                    tumor, normal = token.split("_vs_", 1)
                    samples.add(tumor)
                    samples.add(normal)
                    patients_from_pair_files.add(patient_from_sample(tumor))
                elif token.endswith(("T", "N", "P")):
                    samples.add(token)
                else:
                    patients_from_pair_files.add(token)

    pairs = []
    by_patient = defaultdict(set)
    for sample in samples:
        by_patient[patient_from_sample(sample)].add(sample)

    for patient in sorted(set(by_patient) | patients_from_pair_files):
        ss = by_patient.get(patient, set())
        tumors = sorted([s for s in ss if s.endswith("T")])
        normals = sorted([s for s in ss if s.endswith(("N", "P"))])
        if tumors and normals:
            tumor = tumors[0]
            normal = normals[0]
        else:
            tumor = patient + "T"
            normal = patient + "N"
        pairs.append({"pair_id": f"{tumor}_vs_{normal}", "tumor_id": tumor, "normal_id": normal, "patient": patient})
    return pairs


def collect_vcfs(result_root):
    files_by_tool = {tool: [] for tool in TOOLS}
    for p in Path(result_root).rglob("*.vcf"):
        if ".ipynb_checkpoints" in p.parts:
            continue
        tool = infer_tool(p)
        if tool in files_by_tool:
            files_by_tool[tool].append(p)
    for p in Path(result_root).rglob("*.vcf.gz"):
        if ".ipynb_checkpoints" in p.parts:
            continue
        tool = infer_tool(p)
        if tool in files_by_tool:
            files_by_tool[tool].append(p)
    for tool in TOOLS:
        files_by_tool[tool] = sorted(set(files_by_tool[tool]))
    return files_by_tool


def index_vcfs(files_by_tool):
    idx = {tool: defaultdict(list) for tool in TOOLS}
    for tool, paths in files_by_tool.items():
        for p in paths:
            for token in candidate_tokens(p):
                idx[tool][token].append(p)
                idx[tool][patient_from_sample(token)].append(p)
    return idx


def choose_vcf(tool_index, tool, pair_id, tumor_id, normal_id, patient):
    candidates = []
    for key in [pair_id, tumor_id, patient, normal_id]:
        candidates.extend(tool_index.get(tool, {}).get(key, []))

    if not candidates:
        return "NA"

    pair_l = pair_id.lower()
    tumor_l = tumor_id.lower()
    normal_l = normal_id.lower()
    patient_l = patient.lower()

    def priority(path):
        s = str(path).lower()
        name = Path(path).name.lower()
        score = 1000

        # Best: explicit paired somatic result, e.g. 1866277T_vs_1866277N_somatic.bnd_reclass.vcf
        if pair_l in s:
            score -= 500
        if "somatic" in name:
            score -= 200

        # Never prefer raw single-sample files for paired somatic UpSet.
        if "_raw" in name or ".raw" in name:
            score += 1000

        # Prefer tumor-labelled file over normal-labelled file when pair-level name is absent.
        if tumor_l in name:
            score -= 80
        if normal_l in name:
            score += 80

        # Tool-specific preferences.
        if tool == "gridss":
            if "pon_filtered" in s:
                score -= 100
            if "pon_removed" in s:
                score += 500
            if "gripss" in s:
                score -= 20

        # Svaba paired result may be patient-level:
        # 1866277.svaba.somatic.sv.bnd_reclass.vcf
        if tool == "svaba" and patient_l in name and "somatic" in name:
            score -= 150

        if "bnd_reclass" in s:
            score -= 10
        if s.endswith(".vcf.gz"):
            score -= 2

        return (score, len(str(path)), str(path))

    candidates = sorted(set(candidates), key=priority)
    return str(candidates[0])


def alt_remote(alt):
    m = BND_RE.search(alt)
    if not m:
        return None, None
    return chrom_norm(m.group(1), keep_chr=True), int(m.group(2))


def count_vcf_filters(path):
    total = 0
    pass_records = 0
    nonpass_records = 0
    if path == "NA" or not Path(path).exists():
        return total, pass_records, nonpass_records
    with open_text(path) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 8:
                continue
            total += 1
            filt = fields[6]
            if filt in {"PASS", "."}:
                pass_records += 1
            else:
                nonpass_records += 1
    return total, pass_records, nonpass_records


def parse_vcf_records(path, tool, pair, pass_only=True):
    if path == "NA" or not Path(path).exists():
        return []
    out = []
    with open_text(path) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 8:
                continue
            chrom, pos, rec_id, ref, alt, qual, filt, info_s = fields[:8]
            if pass_only and filt not in {"PASS", "."}:
                continue
            info = parse_info(info_s)
            svtype = extract_svtype(info_s, alt, info)
            pos1 = to_int(pos)
            if pos1 is None:
                continue

            chrom1 = chrom_norm(chrom, keep_chr=True)
            chrom2 = chrom_norm(info.get("CHR2") or info.get("CHROM2") or info.get("ENDCHR") or chrom1, keep_chr=True)
            pos2 = to_int(info.get("END") or info.get("POS2") or info.get("ENDPOS"))

            if svtype in {"BND", "TRA"}:
                a_chrom2, a_pos2 = alt_remote(alt)
                if a_chrom2:
                    chrom2 = a_chrom2
                if a_pos2:
                    pos2 = a_pos2

            if pos2 is None:
                svlen = to_int(info.get("SVLEN"))
                if svlen is not None and svlen != 0:
                    pos2 = pos1 + abs(svlen)
                else:
                    pos2 = pos1

            start = min(pos1, pos2) if chrom_norm(chrom1, False) == chrom_norm(chrom2, False) else pos1
            end = max(pos1, pos2) if chrom_norm(chrom1, False) == chrom_norm(chrom2, False) else pos2
            member_id = f"{tool}:{rec_id or chrom1 + ':' + str(pos1)}"
            out.append({
                "tool": tool,
                "pair_id": pair["pair_id"],
                "tumor_id": pair["tumor_id"],
                "normal_id": pair["normal_id"],
                "chrom1": chrom1,
                "pos1": pos1,
                "chrom2": chrom2,
                "pos2": pos2,
                "start": start,
                "end": end,
                "svtype": svtype,
                "member_id": member_id,
            })
    return out


def event_distance(a, b):
    same_chrom_pair = (
        chrom_norm(a["chrom1"], False) == chrom_norm(b["chrom1"], False)
        and chrom_norm(a["chrom2"], False) == chrom_norm(b["chrom2"], False)
    )
    swapped_chrom_pair = (
        chrom_norm(a["chrom1"], False) == chrom_norm(b["chrom2"], False)
        and chrom_norm(a["chrom2"], False) == chrom_norm(b["chrom1"], False)
    )
    if same_chrom_pair:
        return max(abs(a["pos1"] - b["pos1"]), abs(a["pos2"] - b["pos2"]))
    if swapped_chrom_pair:
        return max(abs(a["pos1"] - b["pos2"]), abs(a["pos2"] - b["pos1"]))
    return None


def compatible(a, b, max_dist, same_type_only):
    if same_type_only and a["svtype"] != b["svtype"]:
        return False
    dist = event_distance(a, b)
    return dist is not None and dist <= max_dist


def merge_events(records, max_dist, same_type_only):
    clusters = []
    for rec in sorted(records, key=lambda r: (chrom_key(r["chrom1"]), r["pos1"], chrom_key(r["chrom2"]), r["pos2"], r["svtype"])):
        hit = None
        for cluster in clusters:
            if any(compatible(rec, old, max_dist, same_type_only) for old in cluster):
                hit = cluster
                break
        if hit is None:
            clusters.append([rec])
        else:
            hit.append(rec)
    return clusters


def cluster_row(pair, idx, cluster):
    tools = sorted({r["tool"] for r in cluster}, key=TOOLS.index)
    svtypes = [r["svtype"] for r in cluster]
    svtype = max(set(svtypes), key=lambda x: (svtypes.count(x), -svtypes.index(x)))
    chrom1 = cluster[0]["chrom1"]
    chrom2 = cluster[0]["chrom2"]
    pos1 = int(round(sum(r["pos1"] for r in cluster) / len(cluster)))
    pos2 = int(round(sum(r["pos2"] for r in cluster) / len(cluster)))
    same = chrom_norm(chrom1, False) == chrom_norm(chrom2, False)
    start = min([r["start"] for r in cluster]) if same else pos1
    end = max([r["end"] for r in cluster]) if same else pos2
    row = {
        "cluster_id": f"SV{idx:06d}",
        "pair_id": pair["pair_id"],
        "tumor_id": pair["tumor_id"],
        "normal_id": pair["normal_id"],
        "chrom1": chrom1,
        "pos1": pos1,
        "chrom2": chrom2,
        "pos2": pos2,
        "start": start,
        "end": end,
        "svtype": svtype,
        "support_n": len(tools),
        "tools": ",".join(tools),
    }
    for tool in TOOLS:
        row[tool] = "1" if tool in tools else "0"
    row["member_record_count"] = len(cluster)
    row["member_record_ids"] = ",".join(r["member_id"] for r in cluster)
    return row


def write_manifest(path, pairs, tool_index):
    cols = ["pair_id", "tumor_id", "normal_id"] + [f"{tool}_vcf" for tool in TOOLS]
    with open(path, "wt") as out:
        out.write("\t".join(cols) + "\n")
        for pair in pairs:
            row = {
                "pair_id": pair["pair_id"],
                "tumor_id": pair["tumor_id"],
                "normal_id": pair["normal_id"],
            }
            for tool in TOOLS:
                row[f"{tool}_vcf"] = choose_vcf(tool_index, tool, pair["pair_id"], pair["tumor_id"], pair["normal_id"], pair["patient"])
            out.write("\t".join(str(row[c]) for c in cols) + "\n")


def main():
    p = argparse.ArgumentParser(description="Build six-tool SV UpSet manifest and merged event tables from BND-reclassified VCFs.")
    p.add_argument("--result-root", default="/mnt/home/ygjx/chenkejin/sv_tools_results_bnd_reclass")
    p.add_argument("--outdir", default="/mnt/home/ygjx/chenkejin/80_upset")
    p.add_argument("--pair-list", default=None, help="Optional TSV with tumor_id and normal_id columns.")
    p.add_argument("--max-distance", type=int, default=1000, help="Max breakpoint distance for merging events across tools.")
    p.add_argument("--same-type-only", action="store_true", help="Merge only records with the same SVTYPE.")
    p.add_argument("--include-filtered", action="store_true", help="Include non-PASS VCF records. Default: use PASS or . only.")
    args = p.parse_args()

    outdir = Path(args.outdir)
    events_dir = outdir / "events"
    events_dir.mkdir(parents=True, exist_ok=True)

    files_by_tool = collect_vcfs(args.result_root)
    tool_index = index_vcfs(files_by_tool)
    pairs = read_pair_list(args.pair_list) if args.pair_list else discover_pairs(files_by_tool)
    pairs = sorted(pairs, key=lambda x: x["pair_id"])

    manifest = outdir / "80_pairs.six_sv_tools.manifest.new.tsv"
    write_manifest(manifest, pairs, tool_index)

    event_cols = [
        "cluster_id", "pair_id", "tumor_id", "normal_id", "chrom1", "pos1", "chrom2", "pos2",
        "start", "end", "svtype", "support_n", "tools",
        "cue", "lumpy", "gridss", "manta", "delly", "svaba",
        "member_record_count", "member_record_ids",
    ]
    summary = outdir / "build_events.summary.tsv"
    audit = outdir / "build_events.vcf_filter_audit.tsv"
    with open(summary, "wt") as s, open(audit, "wt") as a:
        s.write("pair_id\ttumor_id\tnormal_id\traw_records\tmerged_events\toutput\n")
        a.write("pair_id\ttumor_id\tnormal_id\ttool\tvcf\ttotal_records\tpass_records\tnonpass_records\tused_records\tmode\n")
        for pair in pairs:
            records = []
            for tool in TOOLS:
                vcf = choose_vcf(tool_index, tool, pair["pair_id"], pair["tumor_id"], pair["normal_id"], pair["patient"])
                total_n, pass_n, nonpass_n = count_vcf_filters(vcf)
                tool_records = parse_vcf_records(vcf, tool, pair, pass_only=not args.include_filtered)
                records.extend(tool_records)
                mode = "ALL_RECORDS" if args.include_filtered else "PASS_ONLY"
                a.write(
                    f"{pair['pair_id']}\t{pair['tumor_id']}\t{pair['normal_id']}\t{tool}\t{vcf}\t"
                    f"{total_n}\t{pass_n}\t{nonpass_n}\t{len(tool_records)}\t{mode}\n"
                )
            clusters = merge_events(records, args.max_distance, args.same_type_only)
            rows = [cluster_row(pair, i, c) for i, c in enumerate(clusters, 1)]
            out_path = events_dir / f"{pair['pair_id']}.merged_sv_events.tsv"
            with open(out_path, "wt") as out:
                out.write("\t".join(event_cols) + "\n")
                for row in rows:
                    out.write("\t".join(str(row.get(c, "")) for c in event_cols) + "\n")
            s.write(f"{pair['pair_id']}\t{pair['tumor_id']}\t{pair['normal_id']}\t{len(records)}\t{len(rows)}\t{out_path}\n")

    print(f"[DONE] pairs={len(pairs)}")
    print(f"[MANIFEST] {manifest}")
    print(f"[EVENTS] {events_dir}")
    print(f"[SUMMARY] {summary}")
    print(f"[FILTER_AUDIT] {audit}")


if __name__ == "__main__":
    main()

### 运行：

In [ ]:
python /mnt/home/ygjx/chenkejin/80_upset/scripts/build_sv_upset_events_from_bnd_annotated_vcfs.py \ 
  --result-root /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_annotated \
  --outdir /mnt/home/ygjx/chenkejin/80_upset \
  --pair-list /mnt/home/ygjx/chenkejin/80_upset/80_pairs.normal_tumor.tsv \
  --max-distance 1000 --same-type-only

### summarize_80_pairs_sv_by_tool.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import gzip
import re
from collections import Counter
from pathlib import Path


TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]
BND_RE = re.compile(r"[\[\]]([^:\[\]]+):([0-9]+)[\[\]]")


def open_text(path):
    path = str(path)
    if path.endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")


def parse_info(info):
    out = {}
    if not info or info == ".":
        return out
    for item in info.split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" in item:
            key, value = item.split("=", 1)
            out[key.strip()] = value.strip()
        else:
            out[item] = True
    return out


def norm_chrom(chrom):
    chrom = str(chrom or "").strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    chrom = chrom.upper()
    if chrom == "M":
        chrom = "MT"
    return chrom


def chrom_sort_key(chrom):
    chrom = norm_chrom(chrom)
    if chrom.isdigit():
        return (0, int(chrom))
    if chrom == "X":
        return (0, 23)
    if chrom == "Y":
        return (0, 24)
    if chrom == "MT":
        return (0, 25)
    return (1, chrom)


def to_int(value, default=None):
    try:
        if value in {None, "", ".", "NA"}:
            return default
        return int(float(str(value).split(",")[0]))
    except Exception:
        return default


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    mapping = {
        "DELETION": "DEL",
        "DUPLICATION": "DUP",
        "INVERSION": "INV",
        "INSERTION": "INS",
        "TRANSLOCATION": "TRA",
        "CTX": "TRA",
        "BREAKEND": "BND",
    }
    return mapping.get(sv, sv)


def extract_svtype(info_text, alt, info=None):
    if info is None:
        info = {}
    if "BNDINF_SVTYPE" in info:
        sv = norm_svtype(info.get("BNDINF_SVTYPE"))
        if sv != "UNKNOWN":
            return sv
    
    for key in ("SVTYPE", "svtype", "SvType"):
        if key in info:
            sv = norm_svtype(info.get(key))
            if sv != "UNKNOWN":
                return sv

    match = re.search(r"(?i)(?:^|;)SVTYPE=([^;\t\r\n ]+)", info_text or "")
    if match:
        sv = norm_svtype(match.group(1))
        if sv != "UNKNOWN":
            return sv

    alt_text = str(alt or "").strip().upper()
    match = re.search(r"<([^<>:,]+)>", alt_text)
    if match:
        sv = norm_svtype(match.group(1))
        if sv != "UNKNOWN":
            return sv

    if "[" in alt_text or "]" in alt_text:
        return "BND"

    return "UNKNOWN"


def bnd_remote(alt):
    match = BND_RE.search(alt or "")
    if not match:
        return None, None
    return norm_chrom(match.group(1)), int(match.group(2))


def length_bin(length, is_trans):
    if is_trans:
        return "TRA_or_interchrom"
    if length is None:
        return "unknown"
    if length < 50:
        return "<50 bp"
    if length < 100:
        return "50-100 bp"
    if length < 1000:
        return "100 bp-1 kb"
    if length < 10000:
        return "1-10 kb"
    if length < 100000:
        return "10-100 kb"
    if length < 1000000:
        return "100 kb-1 Mb"
    return ">=1 Mb"


def get_manifest_rows(manifest):
    with open(manifest, "rt") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))


def parse_vcf(vcf, pair_id, tumor_id, normal_id, tool, pass_only=True):
    records = []
    if not vcf or vcf == "NA" or not Path(vcf).exists():
        return records

    with open_text(vcf) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue

            fields = line.rstrip("\n").split("\t")
            if len(fields) < 8:
                continue

            chrom, pos, rec_id, ref, alt, qual, filt, info_text = fields[:8]
            if pass_only and filt not in {"PASS", "."}:
                continue

            info = parse_info(info_text)
            svtype = extract_svtype(info_text, alt, info)
            chrom1 = norm_chrom(chrom)
            pos1 = to_int(pos)
            if pos1 is None:
                continue

            chrom2 = norm_chrom(info.get("CHR2") or info.get("CHROM2") or info.get("ENDCHR") or chrom1)
            pos2 = to_int(info.get("END") or info.get("POS2") or info.get("ENDPOS"))

            if svtype in {"BND", "TRA", "DEL", "DUP", "INV", "INS"}:
                remote_chrom, remote_pos = bnd_remote(alt)
                if remote_chrom:
                    chrom2 = remote_chrom
                if remote_pos:
                    pos2 = remote_pos

            svlen = to_int(info.get("SVLEN"))

            if svtype == "INS" and svlen in {None, 0}:
                if "[" in alt or "]" in alt:
                    seq = BND_RE.sub("", alt)
                    seq = "".join(c for c in seq.upper() if c in "ACGTN")
                    if seq:
                        svlen = max(0, len(seq) - len(ref))

            if pos2 is None:
                if svlen not in {None, 0}:
                    pos2 = pos1 + abs(svlen)
                else:
                    pos2 = pos1

            is_trans = chrom1 != chrom2 or svtype == "TRA"
            if svlen not in {None, 0}:
                abs_len = abs(svlen)
            elif is_trans:
                abs_len = None
            else:
                abs_len = abs(pos2 - pos1)

            start = min(pos1, pos2) if not is_trans else pos1
            end = max(pos1, pos2) if not is_trans else pos2

            records.append({
                "pair_id": pair_id,
                "tumor_id": tumor_id,
                "normal_id": normal_id,
                "tool": tool,
                "vcf": vcf,
                "record_id": rec_id or f"{chrom1}:{pos1}:{svtype}",
                "chrom1": chrom1,
                "pos1": pos1,
                "chrom2": chrom2,
                "pos2": pos2,
                "start": start,
                "end": end,
                "svtype": svtype,
                "svlen": abs_len if abs_len is not None else "NA",
                "length_bin": length_bin(abs_len, is_trans),
                "relation": "trans" if is_trans else "cis",
                "filter": filt,
            })

    return records


def write_table(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wt", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, delimiter="\t", extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def count_rows(records, keys, count_name="count"):
    counter = Counter(tuple(record[k] for k in keys) for record in records)
    rows = []
    for key_tuple, count in sorted(counter.items()):
        row = {k: v for k, v in zip(keys, key_tuple)}
        row[count_name] = count
        rows.append(row)
    return rows


def chrom_distribution(records):
    counter = Counter()
    for rec in records:
        chroms = {rec["chrom1"], rec["chrom2"]}
        for chrom in chroms:
            counter[(rec["pair_id"], rec["tumor_id"], rec["normal_id"], rec["tool"], chrom)] += 1

    rows = []
    for (pair_id, tumor_id, normal_id, tool, chrom), count in sorted(
        counter.items(),
        key=lambda x: (x[0][0], x[0][3], chrom_sort_key(x[0][4])),
    ):
        rows.append({
            "pair_id": pair_id,
            "tumor_id": tumor_id,
            "normal_id": normal_id,
            "tool": tool,
            "chrom": chrom,
            "sv_event_involvement_count": count,
        })
    return rows


def trans_chr_pair_distribution(records):
    counter = Counter()
    for rec in records:
        if rec["relation"] != "trans":
            continue
        chrom1, chrom2 = sorted([rec["chrom1"], rec["chrom2"]], key=chrom_sort_key)
        counter[(rec["pair_id"], rec["tumor_id"], rec["normal_id"], rec["tool"], chrom1, chrom2, rec["svtype"])] += 1

    rows = []
    for (pair_id, tumor_id, normal_id, tool, chrom1, chrom2, svtype), count in sorted(
        counter.items(),
        key=lambda x: (x[0][0], x[0][3], chrom_sort_key(x[0][4]), chrom_sort_key(x[0][5]), x[0][6]),
    ):
        rows.append({
            "pair_id": pair_id,
            "tumor_id": tumor_id,
            "normal_id": normal_id,
            "tool": tool,
            "chrom1": chrom1,
            "chrom2": chrom2,
            "svtype": svtype,
            "count": count,
        })
    return rows


def main():
    parser = argparse.ArgumentParser(description="Summarize PASS SV records by pair and tool. Tables only; no plotting.")
    parser.add_argument("--manifest", default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv")
    parser.add_argument("--outdir", default="/mnt/home/ygjx/chenkejin/80_sv_summary")
    parser.add_argument("--include-filtered", action="store_true", help="Include non-PASS records. Default uses PASS or . only.")
    args = parser.parse_args()

    outdir = Path(args.outdir)
    tables_dir = outdir / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows = get_manifest_rows(args.manifest)
    all_records = []

    for row in manifest_rows:
        pair_id = row["pair_id"]
        tumor_id = row["tumor_id"]
        normal_id = row["normal_id"]
        for tool in TOOLS:
            vcf = row.get(f"{tool}_vcf", "NA")
            records = parse_vcf(
                vcf=vcf,
                pair_id=pair_id,
                tumor_id=tumor_id,
                normal_id=normal_id,
                tool=tool,
                pass_only=not args.include_filtered,
            )
            all_records.extend(records)

    record_cols = [
        "pair_id", "tumor_id", "normal_id", "tool", "record_id",
        "chrom1", "pos1", "chrom2", "pos2", "start", "end",
        "svtype", "svlen", "length_bin", "relation", "filter", "vcf",
    ]

    write_table(tables_dir / "all_pass_sv_records.normalized.tsv", all_records, record_cols)
    write_table(
        tables_dir / "total_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool"], "total_sv_count"),
        ["pair_id", "tumor_id", "normal_id", "tool", "total_sv_count"],
    )
    write_table(
        tables_dir / "svtype_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool", "svtype"]),
        ["pair_id", "tumor_id", "normal_id", "tool", "svtype", "count"],
    )
    write_table(
        tables_dir / "length_bin_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool", "length_bin"]),
        ["pair_id", "tumor_id", "normal_id", "tool", "length_bin", "count"],
    )
    write_table(
        tables_dir / "cis_trans_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool", "relation"]),
        ["pair_id", "tumor_id", "normal_id", "tool", "relation", "count"],
    )
    write_table(
        tables_dir / "chromosome_distribution.by_pair_tool.tsv",
        chrom_distribution(all_records),
        ["pair_id", "tumor_id", "normal_id", "tool", "chrom", "sv_event_involvement_count"],
    )
    write_table(
        tables_dir / "trans_chromosome_pair_counts.by_pair_tool.tsv",
        trans_chr_pair_distribution(all_records),
        ["pair_id", "tumor_id", "normal_id", "tool", "chrom1", "chrom2", "svtype", "count"],
    )
    write_table(
        tables_dir / "svtype_counts.overall_by_tool.tsv",
        count_rows(all_records, ["tool", "svtype"]),
        ["tool", "svtype", "count"],
    )
    write_table(
        tables_dir / "length_bin_counts.overall_by_tool.tsv",
        count_rows(all_records, ["tool", "length_bin"]),
        ["tool", "length_bin", "count"],
    )
    write_table(
        tables_dir / "cis_trans_counts.overall_by_tool.tsv",
        count_rows(all_records, ["tool", "relation"]),
        ["tool", "relation", "count"],
    )

    print(f"[DONE] records={len(all_records)}")
    print(f"[TABLES] {tables_dir}")
    print("[PLOTS] skipped")


if __name__ == "__main__":
    main()


### 运行：

In [ ]:
python summarize_80_pairs_sv_by_tool.py \
  --manifest /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv \
  --outdir /mnt/home/ygjx/chenkejin/80_sv_summary

## 二、画图

### 1.每对样本的upset图

### plot_upset_v7_svtype_size_from_events.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import math
from collections import Counter, defaultdict
from pathlib import Path


TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]
PLOT_TOOLS = ["gridss", "delly", "manta", "cue", "lumpy", "svaba"]
TOOL_LABELS = {
    "cue": "CUE",
    "lumpy": "LUMPY",
    "gridss": "GRIDSS",
    "manta": "MANTA",
    "delly": "DELLY",
    "svaba": "SVABA",
}
SVTYPE_PRIORITY = ["DEL", "DUP", "INV", "BND", "TRA", "INS", "SGL"]
SIZE_TRACK_EXCLUDED_SVTYPES = {"TRA", "BND", "SGL", "UNKNOWN"}

# Edit colors here directly. The plotted colors will be exactly these values.
SVTYPE_COLORS = {
    "DEL": "#9cc3e6",
    "DUP": "#8096c8",
    "INV": "#8e8fcf",
    "BND": "#6e8fb2",
    "TRA": "#2f4b8f",
    "INS": "#53999d",
    "SGL": "#248d82",
    "UNKNOWN": "#9E9E9E",
}
SIZE_BINS = [
    ("50-100 bp", 50, 100),
    ("100 bp-1 kb", 100, 1_000),
    ("1-10 kb", 1_000, 10_000),
    ("10-100 kb", 10_000, 100_000),
    ("100 kb-1 Mb", 100_000, 1_000_000),
    (">1 Mb", 1_000_000, math.inf),
]


def parse_args():
    parser = argparse.ArgumentParser(
        description="Plot V7-style UpSet figures with SVTYPE stacked bars and aligned SV length track."
    )
    parser.add_argument(
        "--events-dir",
        default="/mnt/home/ygjx/chenkejin/80_upset/events",
        help="Directory containing *.merged_sv_events.tsv from the existing UpSet workflow.",
    )
    parser.add_argument(
        "--manifest",
        default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv",
        help="Manifest used by the existing UpSet workflow. Required for --task-id.",
    )
    parser.add_argument(
        "--outdir",
        default="/mnt/home/ygjx/chenkejin/80_upset/upset_v7_svtype_size",
        help="Output directory.",
    )
    parser.add_argument("--pair-id", default=None, help="Run one pair id, e.g. 462745T_vs_462745N.")
    parser.add_argument("--task-id", type=int, default=None, help="1-based Slurm array task id in manifest.")
    parser.add_argument("--max-intersections", type=int, default=12, help="Number of top intersections to draw.")
    
    parser.add_argument("--png-dpi", type=int, default=220)
    return parser.parse_args()


def import_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        return plt
    except Exception as exc:
        raise SystemExit(
            "Cannot import matplotlib. Install it in the active conda environment, for example:\n"
            "conda install -c conda-forge matplotlib\n"
            f"Original error: {exc}"
        )


def mkdirs(outdir):
    for sub in ["plots", "summary", "tables", "logs"]:
        (outdir / sub).mkdir(parents=True, exist_ok=True)


def read_manifest_pair(manifest, task_id):
    with open(manifest, "rt") as handle:
        header = handle.readline().rstrip("\n").split("\t")
        for line_no, line in enumerate(handle, start=1):
            if line_no != task_id:
                continue
            row = dict(zip(header, line.rstrip("\n").split("\t")))
            return row.get("pair_id") or f"{row.get('tumor_id')}_vs_{row.get('normal_id')}"
    raise SystemExit(f"No manifest row for task id {task_id}: {manifest}")


def event_paths(args):
    events_dir = Path(args.events_dir)
    if args.task_id is not None:
        pair_id = read_manifest_pair(args.manifest, args.task_id)
        return [events_dir / f"{pair_id}.merged_sv_events.tsv"]
    if args.pair_id:
        return [events_dir / f"{args.pair_id}.merged_sv_events.tsv"]
    return sorted(events_dir.glob("*.merged_sv_events.tsv"))


def read_events(path):
    rows = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            rows.append(row)
    return rows


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DEL", "DELETION"}:
        return "DEL"
    if sv in {"DUP", "DUPLICATION", "IDUP", "DUP:TANDEM", "TANDEM_DUPLICATION"}:
         return "DUP"
    if sv in {"INV", "INVERSION"}:
        return "INV"
    if sv in {"INS", "INSERTION"}:
        return "INS"
    if sv in {"TRA", "TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BND", "BREAKEND"}:
        return "BND"
    if sv in {"SGL", "SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def ordered_svtypes(svtype_total):
    present = [sv for sv, count in svtype_total.items() if count > 0]
    priority = {sv: idx for idx, sv in enumerate(SVTYPE_PRIORITY)}
    return sorted(present, key=lambda sv: (priority.get(sv, len(SVTYPE_PRIORITY)), sv))


def svtype_colors(present_svtypes):
    return {
        sv: SVTYPE_COLORS.get(sv, SVTYPE_COLORS["UNKNOWN"])
        for sv in present_svtypes
    }


def to_int(value, default=None):
    try:
        if value in {"", ".", "NA", None}:
            return default
        return int(float(value))
    except Exception:
        return default


def chrom_clean(value):
    chrom = str(value or "").strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    return chrom.upper()


def event_size(row):
    chrom1 = chrom_clean(row.get("chrom1"))
    chrom2 = chrom_clean(row.get("chrom2"))
    if chrom1 and chrom2 and chrom1 != chrom2:
        return None
    start = to_int(row.get("start"))
    end = to_int(row.get("end"))
    pos1 = to_int(row.get("pos1"))
    pos2 = to_int(row.get("pos2"))
    if start is not None and end is not None:
        return abs(end - start) + 1
    if pos1 is not None and pos2 is not None:
        return abs(pos2 - pos1) + 1
    return None


def size_bin_label(size):
    if size is None:
        return None
    for label, low, high in SIZE_BINS:
        if low <= size < high:
            return label
    if size < SIZE_BINS[0][1]:
        return SIZE_BINS[0][0]
    return None


def row_tools(row):
    listed = [x.strip().lower() for x in str(row.get("tools", "")).split(",") if x.strip()]
    tools = [tool for tool in TOOLS if tool in listed]
    if tools:
        return tuple(tools)
    tools = []
    for tool in TOOLS:
        if str(row.get(tool, "0")).strip() == "1":
            tools.append(tool)
    return tuple(tools)


def y_positions():
    return {tool: len(PLOT_TOOLS) - 1 - idx for idx, tool in enumerate(PLOT_TOOLS)}


def summarize(rows, max_intersections):
    combo_counts = Counter()
    combo_svtype_counts = defaultdict(Counter)
    combo_size_svtype_counts = defaultdict(lambda: defaultdict(Counter))
    tool_svtype_counts = {tool: Counter() for tool in TOOLS}
    svtype_total = Counter()
    size_total = Counter()

    for row in rows:
        tools = row_tools(row)
        if not tools:
            continue
        combo = tuple(tool for tool in TOOLS if tool in tools)
        svtype = norm_svtype(row.get("svtype"))
        size_label = size_bin_label(event_size(row))

        combo_counts[combo] += 1
        combo_svtype_counts[combo][svtype] += 1
        svtype_total[svtype] += 1

        if size_label and svtype not in SIZE_TRACK_EXCLUDED_SVTYPES:
            combo_size_svtype_counts[combo][size_label][svtype] += 1
            size_total[size_label] += 1

        for tool in combo:
            tool_svtype_counts[tool][svtype] += 1

    sorted_combos = sorted(
        combo_counts.items(),
        key=lambda x: (-x[1], -len(x[0]), ",".join(x[0])),
    )[:max_intersections]
    return sorted_combos, combo_svtype_counts, combo_size_svtype_counts, tool_svtype_counts, svtype_total, size_total


def write_tables(outdir, pair_id, rows, sorted_combos, combo_svtype_counts, combo_size_svtype_counts, svtype_total, size_total):
    present_svtypes = ordered_svtypes(svtype_total)
    size_track_svtypes = [sv for sv in present_svtypes if sv not in SIZE_TRACK_EXCLUDED_SVTYPES]
    summary_path = outdir / "summary" / f"{pair_id}.upset_v7_svtype_size.summary.tsv"
    tumor_id = rows[0].get("tumor_id", "") if rows else ""
    normal_id = rows[0].get("normal_id", "") if rows else ""
    with open(summary_path, "wt") as out:
        out.write("pair_id\ttumor_id\tnormal_id\tmerged_sv_clusters\tshown_intersections\n")
        out.write(f"{pair_id}\t{tumor_id}\t{normal_id}\t{len(rows)}\t{len(sorted_combos)}\n")

    counts_path = outdir / "tables" / f"{pair_id}.upset_v7_svtype_size.counts.tsv"
    with open(counts_path, "wt") as out:
        out.write("pair_id\tcategory\tintersection_rank\tcombination\tname\tsvtype\tcount\n")
        for sv in present_svtypes:
            out.write(f"{pair_id}\tsvtype_total\tNA\tNA\tNA\t{sv}\t{svtype_total.get(sv, 0)}\n")
        for label, _low, _high in SIZE_BINS:
            out.write(f"{pair_id}\tsize_total\tNA\tNA\t{label}\tNA\t{size_total.get(label, 0)}\n")
        for rank, (combo, value) in enumerate(sorted_combos, start=1):
            combo_label = ",".join(combo)
            out.write(f"{pair_id}\tintersection_total\t{rank}\t{combo_label}\tNA\tNA\t{value}\n")
            for sv in present_svtypes:
                out.write(f"{pair_id}\tintersection_svtype\t{rank}\t{combo_label}\tNA\t{sv}\t{combo_svtype_counts[combo].get(sv, 0)}\n")
            for size_label, _low, _high in SIZE_BINS:
                for sv in size_track_svtypes:
                    out.write(
                        f"{pair_id}\tintersection_size_svtype\t{rank}\t{combo_label}\t{size_label}\t{sv}\t"
                        f"{combo_size_svtype_counts[combo][size_label].get(sv, 0)}\n"
                    )


def draw_svtype_legend(ax_empty, present_svtypes, colors):
    from matplotlib.patches import Patch

    if not present_svtypes:
        return

    handles = [Patch(facecolor=colors[sv], edgecolor="none", label=sv) for sv in present_svtypes]
    ax_empty.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.50, 0.63),
        frameon=False,
        ncol=1,
        fontsize=13,
        handlelength=2.2,
        handleheight=1.25,
        handletextpad=0.9,
        labelspacing=1.0,
        borderaxespad=0,
    )


def draw_top_stacked_intersections(ax_bar, sorted_combos, combo_svtype_counts, present_svtypes, colors):
    totals = [value for _combo, value in sorted_combos]
    x = list(range(len(sorted_combos)))
    bottom = [0] * len(sorted_combos)
    for sv in reversed(present_svtypes):
        vals = [combo_svtype_counts[combo].get(sv, 0) for combo, _value in sorted_combos]
        ax_bar.bar(x, vals, bottom=bottom, color=colors[sv], width=0.56)
        bottom = [a + b for a, b in zip(bottom, vals)]
    max_total = max(totals) if totals else 1
    ax_bar.set_ylim(0, max_total * 1.20 + 1)
    ax_bar.set_ylabel("Intersection size", fontsize=12)
    ax_bar.set_xticks([])
    ax_bar.spines[["right", "top"]].set_visible(False)
    ax_bar.spines["left"].set_linewidth(1.0)
    ax_bar.grid(axis="y", color="#b7b7b7", lw=0.9)
    ax_bar.set_axisbelow(True)
    for xi, total in zip(x, totals):
        ax_bar.text(xi, total + max_total * 0.015, str(total), ha="center", va="bottom", fontsize=10)


def draw_upset_matrix(ax_labels, ax_matrix, sorted_combos):

    navy = "#2f4b8f"

    inactive = "#DCEBF6"

    row_band = "#F4F9FD"
    yp = y_positions()
    x = list(range(len(sorted_combos)))

    for idx, tool in enumerate(PLOT_TOOLS):
        y = yp[tool]
        if idx % 2 == 1:
            ax_labels.axhspan(y - 0.40, y + 0.40, color=row_band, zorder=0)
            ax_matrix.axhspan(y - 0.40, y + 0.40, color=row_band, zorder=0)

    for tool in PLOT_TOOLS:
        ax_matrix.scatter(x, [yp[tool]] * len(x), s=210, color=inactive, edgecolors="none", zorder=1)
    for xi, (combo, _value) in enumerate(sorted_combos):
        ys = [yp[t] for t in combo if t in yp]
        ax_matrix.scatter([xi] * len(ys), ys, s=230, color=navy, edgecolors="none", zorder=3)
        if len(ys) > 1:
            ax_matrix.plot([xi, xi], [min(ys), max(ys)], color=navy, lw=2.4, solid_capstyle="round", zorder=2)

    ax_matrix.set_xlim(-0.7, max(0, len(sorted_combos) - 0.3))
    ax_matrix.set_ylim(-0.7, len(PLOT_TOOLS) - 0.3)
    ax_matrix.set_yticks([])
    ax_matrix.set_xticks([])
    ax_matrix.spines[["right", "top", "left"]].set_visible(False)
    ax_matrix.spines["bottom"].set_linewidth(1.0)

    ax_labels.set_xlim(0, 1)
    ax_labels.set_ylim(-0.7, len(PLOT_TOOLS) - 0.3)
    ax_labels.axis("off")
    label_texts = []
    for tool in PLOT_TOOLS:
        label_texts.append(
            ax_labels.text(0.97, yp[tool], TOOL_LABELS[tool], ha="right", va="center", fontsize=12)
        )
    return label_texts


def draw_left_stacked_tool_bars(ax_set, tool_svtype_counts, present_svtypes, colors):
    row_band = "#F4F9FD"
    yp = y_positions()
    y_ticks = [yp[t] for t in PLOT_TOOLS]
    totals = {tool: sum(tool_svtype_counts.get(tool, Counter()).values()) for tool in PLOT_TOOLS}
    max_total = max(totals.values()) if totals else 1

    for idx, tool in enumerate(PLOT_TOOLS):
        y = yp[tool]
        if idx % 2 == 1:
            ax_set.axhspan(y - 0.40, y + 0.40, color=row_band, zorder=0)

    bottoms = {tool: 0 for tool in PLOT_TOOLS}
    for sv in reversed(present_svtypes):
        values = [tool_svtype_counts.get(tool, Counter()).get(sv, 0) for tool in PLOT_TOOLS]
        lefts = [bottoms[tool] for tool in PLOT_TOOLS]
        ax_set.barh(y_ticks, values, left=lefts, color=colors[sv], height=0.56)
        for tool, value in zip(PLOT_TOOLS, values):
            bottoms[tool] += value

    ax_set.set_ylim(-0.7, len(PLOT_TOOLS) - 0.3)
    ax_set.set_yticks([])
    ax_set.set_xlim(max_total * 1.45 + 1, 0)
    ax_set.spines[["right", "top", "left"]].set_visible(False)
    ax_set.spines["bottom"].set_linewidth(1.0)
    ax_set.grid(axis="x", color="#dddddd", lw=0.7)
    ax_set.set_axisbelow(True)
    for y, tool in zip(y_ticks, PLOT_TOOLS):
        value = totals[tool]
        ax_set.text(value + max_total * 0.03 + 0.5, y, str(value), va="center", ha="right", fontsize=9, clip_on=False, zorder=10)


def restore_v7_left_bar_position(fig, ax_set, label_texts):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    label_left = min(
        fig.transFigure.inverted().transform_bbox(text.get_window_extent(renderer)).x0
        for text in label_texts
    )
    set_pos = ax_set.get_position()
    target_right = label_left - 0.008
    if target_right > set_pos.x1:
        ax_set.set_position([target_right - set_pos.width, set_pos.y0, set_pos.width, set_pos.height])


def draw_downward_size_track(ax_size, sorted_combos, combo_size_svtype_counts, present_svtypes, colors):
    y_step = 1.55
    size_labels = [label for label, _low, _high in SIZE_BINS]
    y_map = {label: idx * y_step for idx, label in enumerate(size_labels)}
    size_track_svtypes = [sv for sv in present_svtypes if sv not in SIZE_TRACK_EXCLUDED_SVTYPES]
    # Small horizontal jitter within each UpSet column.
    # Keep all SVTYPE points inside their own column and avoid overlap.
    max_jitter = 0.30
    if len(size_track_svtypes) == 1:
        sv_offsets = {size_track_svtypes[0]: 0.0}
    else:
        positions = [
            -max_jitter + i * (2 * max_jitter / (len(size_track_svtypes) - 1))
            for i in range(len(size_track_svtypes))
        ] if len(size_track_svtypes) > 1 else []
        sv_offsets = {sv: offset for sv, offset in zip(size_track_svtypes, positions)}

    for xi, (combo, _value) in enumerate(sorted_combos):
        for size_label in size_labels:
            sv_counter = combo_size_svtype_counts[combo][size_label]
            for sv in size_track_svtypes:
                count = sv_counter.get(sv, 0)
                if count <= 0:
                    continue
                point_size = 26 + min(230, count ** 0.50 * 48)
                ax_size.scatter(
                    xi + sv_offsets.get(sv, 0.0),
                    y_map[size_label],
                    s=point_size,
                    color=colors[sv],
                    alpha=0.88,
                    edgecolors="none",
                    linewidth=0,
                    zorder=3,
                )

    ax_size.set_xlim(-0.5, max(0, len(sorted_combos) - 1 + 0.5))
    ax_size.set_ylim(-0.95, (len(size_labels) - 1) * y_step + 0.95)
    ax_size.invert_yaxis()
    ax_size.set_yticks([y_map[label] for label in size_labels])
    ax_size.set_yticklabels(size_labels, fontsize=9)
    ax_size.set_xticks([])
    ax_size.set_ylabel("SV length", fontsize=10)
    ax_size.set_xlabel("Length distribution aligned with UpSet columns", fontsize=10)
    # Vertical separators: same x centers as UpSet columns.
    for xi in range(len(sorted_combos) - 1):
        ax_size.axvline(
            xi + 0.5,
            color="#D0D0D0",
            linestyle="--",
            linewidth=0.7,
            zorder=0,
        )
    ax_size.grid(axis="y", color="#dddddd", lw=0.7)
    ax_size.spines[["right", "top"]].set_visible(False)
    ax_size.spines["bottom"].set_linewidth(1.0)


def plot_pair(outdir, pair_id, rows, args):
    plt = import_matplotlib()
    (
        sorted_combos,
        combo_svtype_counts,
        combo_size_svtype_counts,
        tool_svtype_counts,
        svtype_total,
        size_total,
    ) = summarize(rows, args.max_intersections)
    write_tables(outdir, pair_id, rows, sorted_combos, combo_svtype_counts, combo_size_svtype_counts, svtype_total, size_total)
    present_svtypes = ordered_svtypes(svtype_total)
    colors = svtype_colors(present_svtypes)

    patient = pair_id.replace("_vs_", "_")
    if rows:
        tumor = rows[0].get("tumor_id", "")
        normal = rows[0].get("normal_id", "")
        if tumor and normal:
            if tumor.endswith("T"):
                patient = tumor[:-1]
            else:
                patient = tumor

    fig_width = max(14.5, min(24, 8.2 + 0.58 * max(1, len(sorted_combos))))
    fig = plt.figure(figsize=(fig_width, 11.7), facecolor="white")
    gs = fig.add_gridspec(
        3,
        3,
        width_ratios=[1.28, 1.95, 9.0],
        height_ratios=[3.65, 2.55, 2.85],
        wspace=0.02,
        hspace=0.06,
    )
    ax_empty = fig.add_subplot(gs[0, :2])
    ax_bar = fig.add_subplot(gs[0, 2])
    ax_set = fig.add_subplot(gs[1, 0])
    ax_labels = fig.add_subplot(gs[1, 1])
    ax_matrix = fig.add_subplot(gs[1, 2], sharex=ax_bar)
    ax_empty2 = fig.add_subplot(gs[2, :2])
    ax_size = fig.add_subplot(gs[2, 2], sharex=ax_bar)

    ax_set.set_zorder(4)
    ax_labels.set_zorder(3)
    ax_set.patch.set_alpha(0)
    ax_empty.axis("off")
    ax_empty2.axis("off")
    fig.suptitle(f"SV Callers Concordance - Patient: {patient}", fontsize=19, y=0.985)

    if sorted_combos:
        draw_svtype_legend(ax_empty, present_svtypes, colors)
        draw_top_stacked_intersections(ax_bar, sorted_combos, combo_svtype_counts, present_svtypes, colors)
        label_texts = draw_upset_matrix(ax_labels, ax_matrix, sorted_combos)
        draw_left_stacked_tool_bars(ax_set, tool_svtype_counts, present_svtypes, colors)
        draw_downward_size_track(ax_size, sorted_combos, combo_size_svtype_counts, present_svtypes, colors)
    else:
        ax_bar.text(0.5, 0.5, "No SV events", ha="center", va="center", fontsize=13)
        for ax in [ax_bar, ax_set, ax_labels, ax_matrix, ax_size]:
            ax.axis("off")
        label_texts = []

    fig.subplots_adjust(left=0.085, right=0.985, bottom=0.07, top=0.90)
    if label_texts:
        restore_v7_left_bar_position(fig, ax_set, label_texts)
    png = outdir / "plots" / f"{pair_id}.upset_v7_svtype_size.png"
    fig.savefig(png, dpi=args.png_dpi, bbox_inches="tight")
    plt.close(fig)
    return png


def combine_summaries(outdir):
    files = sorted((outdir / "summary").glob("*.upset_v7_svtype_size.summary.tsv"))
    combined = outdir / "all_pairs.upset_v7_svtype_size.summary.tsv"
    wrote_header = False
    with open(combined, "wt") as out:
        for path in files:
            with open(path, "rt") as handle:
                header = handle.readline()
                body = handle.read()
            if not wrote_header:
                out.write(header)
                wrote_header = True
            if body:
                out.write(body)
    return combined


def main():
    args = parse_args()
    outdir = Path(args.outdir)
    mkdirs(outdir)
    paths = event_paths(args)
    if not paths:
        raise SystemExit(f"No event tables found in {args.events_dir}")

    for path in paths:
        if not path.exists():
            raise SystemExit(f"Event table not found: {path}")
        rows = read_events(path)
        pair_id = rows[0].get("pair_id") if rows else path.name.replace(".merged_sv_events.tsv", "")
        png = plot_pair(outdir, pair_id, rows, args)
        print(f"[DONE] {pair_id}: {png}")

    combined = combine_summaries(outdir)
    print(f"[SUMMARY] {combined}")


if __name__ == "__main__":
    main()
()


### run_upset.sh：

In [ ]:
#!/bin/bash
#SBATCH --job-name=upset_v7_sv
#SBATCH --nodes=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G
#SBATCH --output=/mnt/home/ygjx/chenkejin/80_upset/slurm_logs/upset_v7_%A_%a.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/80_upset/slurm_logs/upset_v7_%A_%a.err

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate wgs_bam_qc

SCRIPT="/mnt/home/ygjx/chenkejin/80_upset/scripts/plot_upset_v7_svtype_size_from_events.py"
MANIFEST="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv"
EVENTS_DIR="/mnt/home/ygjx/chenkejin/80_upset/events"
OUTDIR="/mnt/home/ygjx/chenkejin/80_upset/upset_v7_svtype_size"

mkdir -p "${OUTDIR}" /mnt/home/ygjx/chenkejin/80_upset/slurm_logs

python "${SCRIPT}" \
  --task-id "${SLURM_ARRAY_TASK_ID}" \
  --manifest "${MANIFEST}" \
  --events-dir "${EVENTS_DIR}" \
  --outdir "${OUTDIR}" \
  --max-intersections 12 \
  --png-dpi 220


### 运行：

In [ ]:
N=$(($(wc -l < /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv) - 1))

sbatch --array=1-${N}%20 \
  /mnt/home/ygjx/chenkejin/80_upset/scripts/run_upset.sh

## 2.每个工具识别出的sv总数及类型占比的堆叠柱状图

### plot_svtype_stacked_bar_by_tool.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
from collections import defaultdict, OrderedDict
from pathlib import Path


# =========================
# User-adjustable settings
# =========================

# Tools to plot. One output figure will be generated for each tool.
TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]

# Display names in plot titles and output filenames.
TOOL_LABELS = {
    "cue": "CUE",
    "lumpy": "LUMPY",
    "gridss": "GRIDSS",
    "manta": "MANTA",
    "delly": "DELLY",
    "svaba": "SvABA",
}

# SVTYPE plotting order. Types absent in a given tool are automatically skipped.
SVTYPE_ORDER = ["DEL", "DUP", "INV", "INS", "BND", "TRA", "SGL", "UNKNOWN"]

# Color palette. Edit these hex colors directly if you want another visual style.
# DEL/DUP/INV/INS: interval-like SVs
# BND/TRA/SGL: breakpoint-like SVs
# UNKNOWN: parsing residue, should ideally be absent
SVTYPE_COLORS = {
    "DEL": "#2F6B9A",
    "DUP": "#4A90B8",
    "INV": "#72B7D2",
    "INS": "#A6CEE3",
    "BND": "#6A4C93",
    "TRA": "#9B5DE5",
    "SGL": "#B388EB",
    "UNKNOWN": "#9E9E9E",
}

# Figure layout settings.
FIG_WIDTH = 26
FIG_HEIGHT = 7
DPI = 300
BAR_WIDTH = 0.84
TITLE_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 13
TICK_FONTSIZE = 7
LEGEND_FONTSIZE = 11

# Show every Nth x-axis sample label. Use 1 to show all 80 labels.
X_LABEL_EVERY = 2


def parse_args():
    parser = argparse.ArgumentParser(
        description="Draw one SVTYPE stacked bar chart per SV caller across 80 tumor-normal pairs."
    )
    parser.add_argument(
        "--svtype-table",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/tables/svtype_counts.by_pair_tool.tsv",
        help="Input table generated by summarize_80_pairs_sv_by_tool.py.",
    )
    parser.add_argument(
        "--manifest",
        default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv",
        help="Manifest used only to keep the 80 pairs in a stable order.",
    )
    parser.add_argument(
        "--outdir",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/plots/svtype_stacked_by_tool",
        help="Output directory for six figures and summary tables.",
    )
    parser.add_argument(
        "--format",
        choices=["png", "pdf", "both"],
        default="png",
        help="Output figure format.",
    )
    parser.add_argument(
        "--label-every",
        type=int,
        default=X_LABEL_EVERY,
        help="Show every Nth x-axis sample label. Use 1 to show all labels.",
    )
    return parser.parse_args()


def import_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        return plt
    except Exception as exc:
        raise SystemExit(
            "Cannot import matplotlib. Install it in the active conda environment, for example:\n"
            "conda install -c conda-forge matplotlib\n"
            f"Original error: {exc}"
        )


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DEL", "DELETION"}:
        return "DEL"
    if sv in {"DUP", "DUPLICATION", "IDUP", "DUP:TANDEM", "TANDEM_DUPLICATION"}:
        return "DUP"
    if sv in {"INV", "INVERSION"}:
        return "INV"
    if sv in {"INS", "INSERTION"}:
        return "INS"
    if sv in {"TRA", "TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BND", "BREAKEND"}:
        return "BND"
    if sv in {"SGL", "SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def read_pair_order(manifest):
    path = Path(manifest)
    if not path.exists():
        return []

    pairs = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            pair_id = row.get("pair_id")
            if not pair_id:
                tumor = row.get("tumor_id", "")
                normal = row.get("normal_id", "")
                pair_id = f"{tumor}_vs_{normal}"
            if pair_id:
                pairs.append(pair_id)
    return pairs


def read_svtype_counts(path):
    # counts[tool][pair_id][svtype] = count
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    pair_seen = OrderedDict()

    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        required = {"pair_id", "tool", "svtype", "count"}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise SystemExit(f"Missing required columns in {path}: {', '.join(sorted(missing))}")

        for row in reader:
            pair_id = row["pair_id"]
            tool = row["tool"].strip().lower()
            svtype = norm_svtype(row["svtype"])
            count = int(float(row["count"]))

            counts[tool][pair_id][svtype] += count
            pair_seen[pair_id] = True

    return counts, list(pair_seen.keys())


def ordered_svtypes_for_tool(tool_counts):
    present = set()
    for pair_counts in tool_counts.values():
        present.update(sv for sv, count in pair_counts.items() if count > 0)

    priority = {sv: idx for idx, sv in enumerate(SVTYPE_ORDER)}
    return sorted(present, key=lambda sv: (priority.get(sv, len(SVTYPE_ORDER)), sv))


def write_tool_matrix(outdir, tool, pair_order, svtypes, tool_counts):
    out = Path(outdir) / f"{tool}.svtype_counts.matrix.tsv"
    with open(out, "wt") as handle:
        handle.write("pair_id\t" + "\t".join(svtypes) + "\ttotal\n")
        for pair_id in pair_order:
            values = [tool_counts.get(pair_id, {}).get(sv, 0) for sv in svtypes]
            handle.write(
                pair_id + "\t" +
                "\t".join(str(v) for v in values) + "\t" +
                str(sum(values)) + "\n"
            )
    return out


def plot_tool(plt, outdir, tool, pair_order, svtypes, tool_counts, output_format, label_every):
    x = list(range(len(pair_order)))
    bottom = [0] * len(pair_order)

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), facecolor="white")

    for sv in svtypes:
        values = [tool_counts.get(pair_id, {}).get(sv, 0) for pair_id in pair_order]
        if not any(values):
            continue
        ax.bar(
            x,
            values,
            bottom=bottom,
            width=BAR_WIDTH,
            color=SVTYPE_COLORS.get(sv, "#777777"),
            edgecolor="none",
            label=sv,
        )
        bottom = [b + v for b, v in zip(bottom, values)]

    label = TOOL_LABELS.get(tool, tool.upper())
    ax.set_title(f"{label}: SVTYPE distribution across 80 tumor-normal pairs", fontsize=TITLE_FONTSIZE, pad=16)
    ax.set_ylabel("PASS SV count", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_xlabel("Tumor-normal pair", fontsize=AXIS_LABEL_FONTSIZE)

    shown_labels = []
    for i, pair_id in enumerate(pair_order):
        shown_labels.append(pair_id if label_every > 0 and i % label_every == 0 else "")

    ax.set_xticks(x)
    ax.set_xticklabels(
        shown_labels,
        rotation=60,
        ha="right",
        rotation_mode="anchor",
        fontsize=TICK_FONTSIZE,
    )
    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="y", color="#D8D8D8", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    legend = ax.legend(
        title=None,
        frameon=False,
        ncol=min(max(len(svtypes), 1), 8),
        fontsize=LEGEND_FONTSIZE,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
    )
    if legend:
        legend_handles = getattr(legend, "legend_handles", getattr(legend, "legendHandles", []))
        for handle in legend_handles:
            handle.set_linewidth(0)

    total_max = max(bottom) if bottom else 0
    ax.set_ylim(0, total_max * 1.15 + 1)

    fig.tight_layout()

    out_paths = []
    if output_format in {"png", "both"}:
        png = Path(outdir) / f"{tool}.svtype_stacked_bar.png"
        fig.savefig(png, dpi=DPI, bbox_inches="tight")
        out_paths.append(png)
    if output_format in {"pdf", "both"}:
        pdf = Path(outdir) / f"{tool}.svtype_stacked_bar.pdf"
        fig.savefig(pdf, bbox_inches="tight")
        out_paths.append(pdf)

    plt.close(fig)
    return out_paths


def main():
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    plt = import_matplotlib()
    counts, table_pair_order = read_svtype_counts(args.svtype_table)
    manifest_pair_order = read_pair_order(args.manifest)

    # Prefer manifest order. If the manifest is unavailable, fall back to table order.
    pair_order = [p for p in manifest_pair_order if p in set(table_pair_order)] or table_pair_order

    if not pair_order:
        raise SystemExit("No pairs found. Check input table and manifest.")

    all_outputs = []
    for tool in TOOLS:
        tool_counts = counts.get(tool, {})
        svtypes = ordered_svtypes_for_tool(tool_counts)
        if not svtypes:
            print(f"[SKIP] {tool}: no records")
            continue

        matrix_path = write_tool_matrix(outdir, tool, pair_order, svtypes, tool_counts)
        fig_paths = plot_tool(
            plt=plt,
            outdir=outdir,
            tool=tool,
            pair_order=pair_order,
            svtypes=svtypes,
            tool_counts=tool_counts,
            output_format=args.format,
            label_every=max(1, args.label_every),
        )

        print(f"[DONE] {tool}: matrix={matrix_path}")
        for path in fig_paths:
            print(f"[FIG] {path}")
        all_outputs.extend(fig_paths)

    print(f"[OUTDIR] {outdir}")
    print(f"[PAIR_N] {len(pair_order)}")
    print(f"[FIG_N] {len(all_outputs)}")


if __name__ == "__main__":
    main()


### 运行：

In [ ]:
python plot_svtype_stacked_bar_by_tool.py \
  --svtype-table /mnt/home/ygjx/chenkejin/80_sv_summary/tables/svtype_counts.by_pair_tool.tsv \
  --manifest /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv \
  --outdir /mnt/home/ygjx/chenkejin/80_sv_summary/plots/svtype_stacked_by_tool \
  --format both \
  --label-every 1

## 3.每对样本的sv长度分布折线图

### plot_sv_length_distribution_lines_by_pair.py

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import math
import re
from collections import defaultdict, OrderedDict
from pathlib import Path


# =========================
# User-adjustable settings
# =========================

# Fixed caller order in every figure.
TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]

TOOL_LABELS = {
    "cue": "CUE",
    "lumpy": "LUMPY",
    "gridss": "GRIDSS",
    "manta": "MANTA",
    "delly": "DELLY",
    "svaba": "SvABA",
}

# Colorblind-friendly scientific palette.
# Change these hex values if you want another style.
TOOL_COLORS = {
    "cue": "#0072B2",     # blue
    "lumpy": "#E69F00",   # orange
    "gridss": "#009E73",  # green
    "manta": "#D55E00",   # vermillion
    "delly": "#CC79A7",   # reddish purple
    "svaba": "#56B4E9",   # sky blue
}

# SV length bins. Length is calculated as abs(SVLEN), or abs(POS2/POS1) fallback.
# Upper bound is inclusive.
LENGTH_BINS = [
    ("50-100 bp", 50, 100),
    ("100 bp-1 kb", 101, 1_000),
    ("1-10 kb", 1_001, 10_000),
    ("10-100 kb", 10_001, 100_000),
    ("100 kb-1 Mb", 100_001, 1_000_000),
    (">1 Mb", 1_000_001, math.inf),
]

# These SV classes do not have a reliable one-dimensional genomic length.
# They are excluded from length-distribution plots by default.
EXCLUDE_SVTYPES_FOR_LENGTH = {"BND", "TRA", "SGL", "UNKNOWN"}

# Figure layout settings.
FIG_WIDTH = 11.5
FIG_HEIGHT = 5.8
DPI = 300
LINE_WIDTH = 2.2
MARKER_SIZE = 6
TITLE_FONTSIZE = 14
AXIS_LABEL_FONTSIZE = 12
TICK_FONTSIZE = 10
LEGEND_FONTSIZE = 10
LEGEND_OUTSIDE_RIGHT = True


def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Draw one SV length-distribution line plot per tumor-normal pair. "
            "Each figure has six lines, one for each SV caller."
        )
    )
    parser.add_argument(
        "--records",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/tables/all_pass_sv_records.normalized.tsv",
        help="Input normalized PASS SV record table generated by summarize_80_pairs_sv_by_tool.py.",
    )
    parser.add_argument(
        "--manifest",
        default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv",
        help="Manifest used only to keep pair order stable.",
    )
    parser.add_argument(
        "--outdir",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/plots/sv_length_lines_by_pair",
        help="Output directory for figures and length-count table.",
    )
    parser.add_argument(
        "--format",
        choices=["png", "pdf", "both"],
        default="png",
        help="Output figure format.",
    )
    parser.add_argument(
        "--yscale",
        choices=["linear", "log"],
        default="linear",
        help="Y-axis scale. Use log when a pair has very uneven counts.",
    )
    parser.add_argument(
        "--include-breakpoint-types",
        action="store_true",
        help="Also include BND/TRA/SGL/UNKNOWN if a usable length exists. Not recommended.",
    )
    return parser.parse_args()


def import_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        return plt
    except Exception as exc:
        raise SystemExit(
            "Cannot import matplotlib. Install it in the active conda environment, for example:\n"
            "conda install -c conda-forge matplotlib\n"
            f"Original error: {exc}"
        )


def normalize_key(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def choose_column(fieldnames, candidates, required=True):
    norm_to_real = {normalize_key(x): x for x in fieldnames}
    for c in candidates:
        key = normalize_key(c)
        if key in norm_to_real:
            return norm_to_real[key]
    if required:
        raise SystemExit(
            "Cannot find required column. Tried: "
            + ", ".join(candidates)
            + "\nAvailable columns: "
            + ", ".join(fieldnames)
        )
    return None


def to_int(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s in {"", ".", "NA", "nan", "None"}:
            return None
        return int(float(s))
    except Exception:
        return None


def norm_chr(value):
    s = str(value or "").strip()
    if not s:
        return ""
    s = re.sub(r"^chr", "", s, flags=re.IGNORECASE)
    return s.upper()


def norm_tool(value):
    return str(value or "").strip().lower()


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DEL", "DELETION"}:
        return "DEL"
    if sv in {"DUP", "DUPLICATION", "IDUP", "DUP:TANDEM", "TANDEM_DUPLICATION"}:
        return "DUP"
    if sv in {"INV", "INVERSION"}:
        return "INV"
    if sv in {"INS", "INSERTION"}:
        return "INS"
    if sv in {"TRA", "TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BND", "BREAKEND"}:
        return "BND"
    if sv in {"SGL", "SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def length_bin(length):
    if length is None or length <= 0:
        return None
    for label, low, high in LENGTH_BINS:
        if low <= length <= high:
            return label
    return None


def read_pair_order(manifest):
    path = Path(manifest)
    if not path.exists():
        return []

    pairs = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            pair_id = row.get("pair_id")
            if not pair_id:
                tumor = row.get("tumor_id", "")
                normal = row.get("normal_id", "")
                pair_id = f"{tumor}_vs_{normal}" if tumor and normal else ""
            if pair_id:
                pairs.append(pair_id)
    return pairs


def infer_length(row, cols):
    # Preferred: explicit SVLEN / length column.
    if cols["length"]:
        val = to_int(row.get(cols["length"]))
        if val is not None:
            return abs(val)

    # Fallback: intra-chromosomal distance.
    chr1 = norm_chr(row.get(cols["chr1"])) if cols["chr1"] else ""
    chr2 = norm_chr(row.get(cols["chr2"])) if cols["chr2"] else chr1
    pos1 = to_int(row.get(cols["pos1"])) if cols["pos1"] else None
    pos2 = to_int(row.get(cols["pos2"])) if cols["pos2"] else None

    if chr1 and chr2 and chr1 == chr2 and pos1 is not None and pos2 is not None:
        return abs(pos2 - pos1)
    return None


def read_and_count(records_path, include_breakpoint_types=False):
    # counts[pair_id][tool][bin_label] = count
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    skipped = defaultdict(int)
    pair_seen = OrderedDict()

    with open(records_path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        fieldnames = reader.fieldnames or []

        cols = {
            "pair": choose_column(fieldnames, ["pair_id", "pair", "pairid"]),
            "tool": choose_column(fieldnames, ["tool", "caller"]),
            "svtype": choose_column(fieldnames, ["svtype", "sv_type", "type"]),
            "length": choose_column(
                fieldnames,
                ["svlen", "sv_len", "sv_length", "length", "size", "abs_svlen", "abs_length"],
                required=False,
            ),
            "chr1": choose_column(fieldnames, ["chrom1", "chr1", "chrom", "chr"], required=False),
            "pos1": choose_column(fieldnames, ["pos1", "start1", "pos", "start"], required=False),
            "chr2": choose_column(fieldnames, ["chrom2", "chr2", "chrom_b", "chr_b"], required=False),
            "pos2": choose_column(fieldnames, ["pos2", "end2", "end", "stop"], required=False),
        }

        for row in reader:
            pair_id = row.get(cols["pair"], "").strip()
            tool = norm_tool(row.get(cols["tool"]))
            svtype = norm_svtype(row.get(cols["svtype"]))
            if not pair_id or tool not in TOOLS:
                skipped["bad_pair_or_tool"] += 1
                continue

            pair_seen[pair_id] = True

            if not include_breakpoint_types and svtype in EXCLUDE_SVTYPES_FOR_LENGTH:
                skipped[f"excluded_{svtype}"] += 1
                continue

            sv_length = infer_length(row, cols)
            bin_label = length_bin(sv_length)
            if bin_label is None:
                skipped["no_usable_length"] += 1
                continue

            counts[pair_id][tool][bin_label] += 1

    return counts, list(pair_seen.keys()), skipped


def write_length_count_table(outdir, pair_order, counts):
    out = Path(outdir) / "sv_length_bin_counts.by_pair_tool.tsv"
    bin_labels = [x[0] for x in LENGTH_BINS]

    with open(out, "wt") as handle:
        handle.write("pair_id\ttool\tlength_bin\tcount\n")
        for pair_id in pair_order:
            for tool in TOOLS:
                for bin_label in bin_labels:
                    handle.write(
                        f"{pair_id}\t{tool}\t{bin_label}\t"
                        f"{counts.get(pair_id, {}).get(tool, {}).get(bin_label, 0)}\n"
                    )
    return out


def write_skip_summary(outdir, skipped):
    out = Path(outdir) / "sv_length_plot.skipped_records.summary.tsv"
    with open(out, "wt") as handle:
        handle.write("reason\tcount\n")
        for reason in sorted(skipped):
            handle.write(f"{reason}\t{skipped[reason]}\n")
    return out


def plot_pair(plt, outdir, pair_id, pair_counts, output_format, yscale):
    bin_labels = [x[0] for x in LENGTH_BINS]
    x = list(range(len(bin_labels)))

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), facecolor="white")

    for tool in TOOLS:
        y = [pair_counts.get(tool, {}).get(label, 0) for label in bin_labels]
        ax.plot(
            x,
            y,
            color=TOOL_COLORS[tool],
            linewidth=LINE_WIDTH,
            marker="o",
            markersize=MARKER_SIZE,
            markerfacecolor=TOOL_COLORS[tool],
            markeredgecolor="white",
            markeredgewidth=0.8,
            label=TOOL_LABELS[tool],
        )

    ax.set_title(f"{pair_id}: SV length distribution by caller", fontsize=TITLE_FONTSIZE, pad=14)
    ax.set_xlabel("SV length bin", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("PASS SV count", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels, rotation=25, ha="right", fontsize=TICK_FONTSIZE)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)

    if yscale == "log":
        ax.set_yscale("symlog", linthresh=1)
        ax.set_ylabel("PASS SV count (symlog)", fontsize=AXIS_LABEL_FONTSIZE)

    ax.grid(axis="y", color="#D8D8D8", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    if LEGEND_OUTSIDE_RIGHT:
        legend = ax.legend(    
            frameon=False,
            fontsize=LEGEND_FONTSIZE,
            ncol=1,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0,
            handlelength=2.4,
            labelspacing=0.9,
        )
    else:
        legend = ax.legend(
            frameon=False,
            fontsize=LEGEND_FONTSIZE,
            ncol=3,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.08),
        )

    if legend:
        for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
            handle.set_linewidth(LINE_WIDTH)

    if LEGEND_OUTSIDE_RIGHT:
        fig.subplots_adjust(left=0.10, right=0.80, bottom=0.20, top=0.88)
    else:
        fig.subplots_adjust(left=0.10, right=0.97, bottom=0.20, top=0.84)

    safe_pair = re.sub(r"[^A-Za-z0-9_.-]+", "_", pair_id)
    out_paths = []
    if output_format in {"png", "both"}:
        png = Path(outdir) / f"{safe_pair}.sv_length_distribution.lines.png"
        fig.savefig(png, dpi=DPI, bbox_inches="tight")
        out_paths.append(png)
    if output_format in {"pdf", "both"}:
        pdf = Path(outdir) / f"{safe_pair}.sv_length_distribution.lines.pdf"
        fig.savefig(pdf, bbox_inches="tight")
        out_paths.append(pdf)

    plt.close(fig)
    return out_paths


def main():
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    plt = import_matplotlib()
    counts, table_pair_order, skipped = read_and_count(
        args.records,
        include_breakpoint_types=args.include_breakpoint_types,
    )
    manifest_pair_order = read_pair_order(args.manifest)

    pair_set = set(table_pair_order)
    pair_order = [p for p in manifest_pair_order if p in pair_set] or table_pair_order

    if not pair_order:
        raise SystemExit("No pairs found. Check input table and manifest.")

    count_table = write_length_count_table(outdir, pair_order, counts)
    skip_table = write_skip_summary(outdir, skipped)

    fig_n = 0
    for pair_id in pair_order:
        fig_paths = plot_pair(
            plt=plt,
            outdir=outdir,
            pair_id=pair_id,
            pair_counts=counts.get(pair_id, {}),
            output_format=args.format,
            yscale=args.yscale,
        )
        fig_n += len(fig_paths)

    print(f"[DONE] pairs={len(pair_order)}")
    print(f"[FIGURES] {fig_n}")
    print(f"[COUNT_TABLE] {count_table}")
    print(f"[SKIP_TABLE] {skip_table}")
    print(f"[OUTDIR] {outdir}")


if __name__ == "__main__":
    main()


### 运行：

In [ ]:
python plot_sv_length_distribution_lines_by_pair.py \
  --records /mnt/home/ygjx/chenkejin/80_sv_summary/tables/all_pass_sv_records.normalized.tsv \
  --manifest /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv \
  --outdir /mnt/home/ygjx/chenkejin/80_sv_summary/plots/sv_length_lines_by_pair \
  --format both

## 4.每个工具识别出的sv总数及不同长度占比的堆叠柱状图

### plot_sv_length_stacked_bar_by_tool.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import math
import re
from collections import defaultdict, OrderedDict
from pathlib import Path


# =========================
# User-adjustable settings
# =========================

# One stacked bar figure will be generated for each tool.
TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]

TOOL_LABELS = {
    "cue": "CUE",
    "lumpy": "LUMPY",
    "gridss": "GRIDSS",
    "manta": "MANTA",
    "delly": "DELLY",
    "svaba": "SvABA",
}

# SV length bins. The upper bound is inclusive.
# Length is calculated from abs(SVLEN) first; if absent, use same-chromosome distance.
LENGTH_BINS = [
    ("50-100 bp", 50, 100),
    ("100 bp-1 kb", 101, 1_000),
    ("1-10 kb", 1_001, 10_000),
    ("10-100 kb", 10_001, 100_000),
    ("100 kb-1 Mb", 100_001, 1_000_000),
    (">1 Mb", 1_000_001, math.inf),
]

# Sequential blue palette from light to dark.
# Small SVs use lighter colors; larger SVs use darker colors.
LENGTH_BIN_COLORS = {
    "50-100 bp": "#D6EAF8",
    "100 bp-1 kb": "#AED6F1",
    "1-10 kb": "#5DADE2",
    "10-100 kb": "#2E86C1",
    "100 kb-1 Mb": "#1B4F72",
    ">1 Mb": "#0B2545",
}

# Breakpoint-like SV classes do not have a robust one-dimensional genomic length.
# They are excluded from length-distribution plots by default.
EXCLUDE_SVTYPES_FOR_LENGTH = {"BND", "TRA", "SGL", "UNKNOWN"}

# Figure layout settings.
FIG_WIDTH = 32
FIG_HEIGHT = 8
DPI = 300
BAR_WIDTH = 0.84
TITLE_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 13
TICK_FONTSIZE = 6
LEGEND_FONTSIZE = 11

# Show every Nth x-axis sample label. Use 1 to show all 80 labels.
X_LABEL_EVERY = 1
X_LABEL_ROTATION = 60


def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Draw one SV length stacked bar chart per SV caller across 80 tumor-normal pairs."
        )
    )
    parser.add_argument(
        "--records",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/tables/all_pass_sv_records.normalized.tsv",
        help="Input normalized PASS SV record table generated by summarize_80_pairs_sv_by_tool.py.",
    )
    parser.add_argument(
        "--manifest",
        default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv",
        help="Manifest used only to keep the 80 pairs in a stable order.",
    )
    parser.add_argument(
        "--outdir",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/plots/sv_length_stacked_by_tool",
        help="Output directory for six figures and summary matrices.",
    )
    parser.add_argument(
        "--format",
        choices=["png", "pdf", "both"],
        default="png",
        help="Output figure format.",
    )
    parser.add_argument(
        "--label-every",
        type=int,
        default=X_LABEL_EVERY,
        help="Show every Nth x-axis sample label. Use 1 to show all 80 labels.",
    )
    parser.add_argument(
        "--include-breakpoint-types",
        action="store_true",
        help="Also include BND/TRA/SGL/UNKNOWN if a usable length exists. Not recommended.",
    )
    return parser.parse_args()


def import_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        return plt
    except Exception as exc:
        raise SystemExit(
            "Cannot import matplotlib. Install it in the active conda environment, for example:\n"
            "conda install -c conda-forge matplotlib\n"
            f"Original error: {exc}"
        )


def normalize_key(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def choose_column(fieldnames, candidates, required=True):
    norm_to_real = {normalize_key(x): x for x in fieldnames}
    for candidate in candidates:
        key = normalize_key(candidate)
        if key in norm_to_real:
            return norm_to_real[key]
    if required:
        raise SystemExit(
            "Cannot find required column. Tried: "
            + ", ".join(candidates)
            + "\nAvailable columns: "
            + ", ".join(fieldnames)
        )
    return None


def to_int(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s in {"", ".", "NA", "nan", "None"}:
            return None
        return int(float(s))
    except Exception:
        return None


def norm_chr(value):
    s = str(value or "").strip()
    if not s:
        return ""
    s = re.sub(r"^chr", "", s, flags=re.IGNORECASE)
    return s.upper()


def norm_tool(value):
    return str(value or "").strip().lower()


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DEL", "DELETION"}:
        return "DEL"
    if sv in {"DUP", "DUPLICATION", "IDUP", "DUP:TANDEM", "TANDEM_DUPLICATION"}:
        return "DUP"
    if sv in {"INV", "INVERSION"}:
        return "INV"
    if sv in {"INS", "INSERTION"}:
        return "INS"
    if sv in {"TRA", "TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BND", "BREAKEND"}:
        return "BND"
    if sv in {"SGL", "SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def length_bin(length):
    if length is None or length <= 0:
        return None
    for label, low, high in LENGTH_BINS:
        if low <= length <= high:
            return label
    return None


def read_pair_order(manifest):
    path = Path(manifest)
    if not path.exists():
        return []

    pairs = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            pair_id = row.get("pair_id")
            if not pair_id:
                tumor = row.get("tumor_id", "")
                normal = row.get("normal_id", "")
                pair_id = f"{tumor}_vs_{normal}" if tumor and normal else ""
            if pair_id:
                pairs.append(pair_id)
    return pairs


def infer_length(row, cols):
    # Preferred: explicit SVLEN / length column.
    if cols["length"]:
        val = to_int(row.get(cols["length"]))
        if val is not None:
            return abs(val)

    # Fallback: intra-chromosomal endpoint distance.
    chr1 = norm_chr(row.get(cols["chr1"])) if cols["chr1"] else ""
    chr2 = norm_chr(row.get(cols["chr2"])) if cols["chr2"] else chr1
    pos1 = to_int(row.get(cols["pos1"])) if cols["pos1"] else None
    pos2 = to_int(row.get(cols["pos2"])) if cols["pos2"] else None

    if chr1 and chr2 and chr1 == chr2 and pos1 is not None and pos2 is not None:
        return abs(pos2 - pos1)
    return None


def read_and_count(records_path, include_breakpoint_types=False):
    # counts[tool][pair_id][length_bin] = count
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    pair_seen = OrderedDict()
    skipped = defaultdict(int)

    with open(records_path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        fieldnames = reader.fieldnames or []

        cols = {
            "pair": choose_column(fieldnames, ["pair_id", "pair", "pairid"]),
            "tool": choose_column(fieldnames, ["tool", "caller"]),
            "svtype": choose_column(fieldnames, ["svtype", "sv_type", "type"]),
            "length": choose_column(
                fieldnames,
                ["svlen", "sv_len", "sv_length", "length", "size", "abs_svlen", "abs_length"],
                required=False,
            ),
            "chr1": choose_column(fieldnames, ["chrom1", "chr1", "chrom", "chr"], required=False),
            "pos1": choose_column(fieldnames, ["pos1", "start1", "pos", "start"], required=False),
            "chr2": choose_column(fieldnames, ["chrom2", "chr2", "chrom_b", "chr_b"], required=False),
            "pos2": choose_column(fieldnames, ["pos2", "end2", "end", "stop"], required=False),
        }

        for row in reader:
            pair_id = row.get(cols["pair"], "").strip()
            tool = norm_tool(row.get(cols["tool"]))
            svtype = norm_svtype(row.get(cols["svtype"]))

            if not pair_id or tool not in TOOLS:
                skipped["bad_pair_or_tool"] += 1
                continue

            pair_seen[pair_id] = True

            if not include_breakpoint_types and svtype in EXCLUDE_SVTYPES_FOR_LENGTH:
                skipped[f"excluded_{svtype}"] += 1
                continue

            sv_length = infer_length(row, cols)
            bin_label = length_bin(sv_length)
            if bin_label is None:
                skipped["no_usable_length"] += 1
                continue

            counts[tool][pair_id][bin_label] += 1

    return counts, list(pair_seen.keys()), skipped


def write_tool_matrix(outdir, tool, pair_order, tool_counts):
    out = Path(outdir) / f"{tool}.sv_length_bin_counts.matrix.tsv"
    bin_labels = [x[0] for x in LENGTH_BINS]

    with open(out, "wt") as handle:
        handle.write("pair_id\t" + "\t".join(bin_labels) + "\ttotal\n")
        for pair_id in pair_order:
            values = [tool_counts.get(pair_id, {}).get(label, 0) for label in bin_labels]
            handle.write(
                pair_id
                + "\t"
                + "\t".join(str(v) for v in values)
                + "\t"
                + str(sum(values))
                + "\n"
            )
    return out


def write_skip_summary(outdir, skipped):
    out = Path(outdir) / "sv_length_stacked_bar.skipped_records.summary.tsv"
    with open(out, "wt") as handle:
        handle.write("reason\tcount\n")
        for reason in sorted(skipped):
            handle.write(f"{reason}\t{skipped[reason]}\n")
    return out


def plot_tool(plt, outdir, tool, pair_order, tool_counts, output_format, label_every):
    bin_labels = [x[0] for x in LENGTH_BINS]
    x = list(range(len(pair_order)))
    bottom = [0] * len(pair_order)

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), facecolor="white")

    for bin_label in bin_labels:
        values = [tool_counts.get(pair_id, {}).get(bin_label, 0) for pair_id in pair_order]
        if not any(values):
            continue
        ax.bar(
            x,
            values,
            bottom=bottom,
            width=BAR_WIDTH,
            color=LENGTH_BIN_COLORS.get(bin_label, "#777777"),
            edgecolor="none",
            label=bin_label,
        )
        bottom = [b + v for b, v in zip(bottom, values)]

    label = TOOL_LABELS.get(tool, tool.upper())
    ax.set_title(
        f"{label}: SV length distribution across 80 tumor-normal pairs",
        fontsize=TITLE_FONTSIZE,
        pad=16,
    )
    ax.set_ylabel("PASS SV count", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_xlabel("Tumor-normal pair", fontsize=AXIS_LABEL_FONTSIZE)

    shown_labels = []
    for i, pair_id in enumerate(pair_order):
        shown_labels.append(pair_id if label_every > 0 and i % label_every == 0 else "")

    ax.set_xticks(x)
    ax.set_xticklabels(
        shown_labels,
        rotation=X_LABEL_ROTATION,
        ha="right",
        fontsize=TICK_FONTSIZE,
    )
    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="y", color="#D8D8D8", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    legend = ax.legend(
        title=None,
        frameon=False,
        ncol=len(bin_labels),
        fontsize=LEGEND_FONTSIZE,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.04),
        handlelength=1.5,
        columnspacing=1.0,
    )
    if legend:
        for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
            handle.set_linewidth(0)

    total_max = max(bottom) if bottom else 0
    ax.set_ylim(0, total_max * 1.15 + 1)

    fig.subplots_adjust(left=0.06, right=0.99, bottom=0.30, top=0.86)

    out_paths = []
    if output_format in {"png", "both"}:
        png = Path(outdir) / f"{tool}.sv_length_stacked_bar.png"
        fig.savefig(png, dpi=DPI, bbox_inches="tight")
        out_paths.append(png)
    if output_format in {"pdf", "both"}:
        pdf = Path(outdir) / f"{tool}.sv_length_stacked_bar.pdf"
        fig.savefig(pdf, bbox_inches="tight")
        out_paths.append(pdf)

    plt.close(fig)
    return out_paths


def main():
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    plt = import_matplotlib()

    counts, table_pair_order, skipped = read_and_count(
        args.records,
        include_breakpoint_types=args.include_breakpoint_types,
    )
    manifest_pair_order = read_pair_order(args.manifest)

    pair_set = set(table_pair_order)
    pair_order = [p for p in manifest_pair_order if p in pair_set] or table_pair_order

    if not pair_order:
        raise SystemExit("No pairs found. Check input table and manifest.")

    skip_table = write_skip_summary(outdir, skipped)

    all_outputs = []
    for tool in TOOLS:
        tool_counts = counts.get(tool, {})
        matrix_path = write_tool_matrix(outdir, tool, pair_order, tool_counts)
        fig_paths = plot_tool(
            plt=plt,
            outdir=outdir,
            tool=tool,
            pair_order=pair_order,
            tool_counts=tool_counts,
            output_format=args.format,
            label_every=max(1, args.label_every),
        )

        print(f"[DONE] {tool}: matrix={matrix_path}")
        for path in fig_paths:
            print(f"[FIG] {path}")
        all_outputs.extend(fig_paths)

    print(f"[SKIP_TABLE] {skip_table}")
    print(f"[OUTDIR] {outdir}")
    print(f"[PAIR_N] {len(pair_order)}")
    print(f"[FIG_N] {len(all_outputs)}")


if __name__ == "__main__":
    main()


### 运行：

In [ ]:
python plot_sv_length_stacked_bar_by_tool.py \
  --records /mnt/home/ygjx/chenkejin/80_sv_summary/tables/all_pass_sv_records.normalized.tsv \
  --manifest /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv \
  --outdir /mnt/home/ygjx/chenkejin/80_sv_summary/plots/sv_length_stacked_by_tool \
  --format both \
  --label-every 1

## 4.每个工具识别出的每对样本sv的染色体分布情况图

### plot_pair_circos_svtype_tool_rings.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import math
import re
from collections import defaultdict, OrderedDict
from pathlib import Path


# =========================
# User-adjustable settings
# =========================

# Six callers shown by color.
TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]

TOOL_LABELS = {
    "cue": "CUE",
    "lumpy": "LUMPY",
    "gridss": "GRIDSS",
    "manta": "MANTA",
    "delly": "DELLY",
    "svaba": "SvABA",
}

# Tool colors. These are colorblind-friendly and distinct enough for circos rings.
TOOL_COLORS = {
    "cue": "#0072B2",
    "lumpy": "#E69F00",
    "gridss": "#009E73",
    "manta": "#D55E00",
    "delly": "#CC79A7",
    "svaba": "#56B4E9",
}

# SVTYPE ring order, outside to inside.
SVTYPE_ORDER = ["DEL", "DUP", "INV", "INS", "TRA", "BND", "SGL", "UNKNOWN"]

# hg38 primary chromosome lengths.
CHR_LENGTHS = OrderedDict([
    ("1", 248956422),
    ("2", 242193529),
    ("3", 198295559),
    ("4", 190214555),
    ("5", 181538259),
    ("6", 170805979),
    ("7", 159345973),
    ("8", 145138636),
    ("9", 138394717),
    ("10", 133797422),
    ("11", 135086622),
    ("12", 133275309),
    ("13", 114364328),
    ("14", 107043718),
    ("15", 101991189),
    ("16", 90338345),
    ("17", 83257441),
    ("18", 80373285),
    ("19", 58617616),
    ("20", 64444167),
    ("21", 46709983),
    ("22", 50818468),
    ("X", 156040895),
    ("Y", 57227415),
])

CHR_ORDER = list(CHR_LENGTHS.keys())

# Overall figure style.
FIG_WIDTH = 10.5
FIG_HEIGHT = 10.5
DPI = 300

# Chromosome ideogram ring.
OUTER_RADIUS = 1.00
CHR_RING_WIDTH = 0.045
CHR_GAP_RAD = 0.014
CHR_LABEL_RADIUS = 1.10

# SVTYPE ring layout.
# V3 uses true annular bands, not single reference circles.
# Each SVTYPE is a band with width; the six tools occupy fixed subtracks inside that band.
SV_BAND_OUTER = 0.91
SV_BAND_WIDTH_MAX = 0.064
SV_BAND_WIDTH_MIN = 0.034
SV_BAND_GAP = 0.012
SV_BAND_TARGET_INNER = 0.52
SV_BAND_EDGE_COLOR = "#9AA3AA"
SV_BAND_EDGE_WIDTH = 0.38
SV_BAND_ALPHA = 0.88
TOOL_TRACK_PADDING = 0.004

# Pale fills distinguish ring bands without competing with tool colors.
# Tool identity is still encoded by the colored ticks and central links.
SVTYPE_BAND_COLORS = {
    "DEL": "#F8FBFF",
    "DUP": "#F5F9FE",
    "INV": "#F2F7FC",
    "INS": "#EFF5FA",
    "TRA": "#F7F5FC",
    "BND": "#F4F1FA",
    "SGL": "#F6F6F6",
    "UNKNOWN": "#FAFAFA",
}

# Marker/tick settings.
# SV events are shown as short radial ticks, not points.
TICK_LENGTH = 0.030
TICK_LINE_WIDTH = 1.05
TICK_ALPHA = 0.90

# Chromosome sector guide lines, from outer ideogram toward the center.
DRAW_CHR_SECTORS = True
SECTOR_INNER_RADIUS = 0.06
SECTOR_OUTER_RADIUS = 0.985
SECTOR_LINE_WIDTH = 0.42
SECTOR_ALPHA = 0.40
SECTOR_COLOR = "#9AA3AA"

# Optional central links for inter-chromosomal TRA/BND events.
DRAW_TRA_BND_LINKS = True
LINK_RADIUS_DEFAULT = 0.64
LINK_RADIUS_MIN = 0.24
LINK_TRACK_MARGIN = 0.075
LINK_ALPHA = 0.56
LINK_WIDTH = 0.82
MAX_LINKS_PER_FIGURE = 1200

# Small "pizza-slice" legend explaining that SVTYPE is encoded by annular bands.
DRAW_SVTYPE_PIZZA_LEGEND = True
PIZZA_CENTER = (1.18, -0.92)
PIZZA_OUTER_RADIUS = 0.22
PIZZA_BAND_WIDTH = 0.018
PIZZA_BAND_GAP = 0.004
PIZZA_THETA1 = 305
PIZZA_THETA2 = 358
PIZZA_LABEL_DY = 0.034

# Text settings.
TITLE_FONTSIZE = 13
SUBTITLE_FONTSIZE = 9
CHR_LABEL_FONTSIZE = 7
SVTYPE_LABEL_FONTSIZE = 8
LEGEND_FONTSIZE = 8


def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Draw one integrated circos plot per tumor-normal pair. "
            "SVTYPE is encoded by annular bands, tools are encoded by color, "
            "and inter-chromosomal TRA/BND events are shown as central links."
        )
    )
    parser.add_argument(
        "--records",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/tables/all_pass_sv_records.normalized.tsv",
        help="Input normalized PASS SV record table generated by summarize_80_pairs_sv_by_tool.py.",
    )
    parser.add_argument(
        "--manifest",
        default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv",
        help="Manifest used only to keep pair order stable.",
    )
    parser.add_argument(
        "--outdir",
        default="/mnt/home/ygjx/chenkejin/80_sv_summary/plots/pair_circos_svtype_tool_rings_v3",
        help="Output directory for 80 integrated circos figures and status table.",
    )
    parser.add_argument(
        "--format",
        choices=["png", "pdf", "both"],
        default="png",
        help="Output figure format.",
    )
    parser.add_argument(
        "--show-empty-rings",
        action="store_true",
        help="Show all SVTYPE rings even when a pair has no records for that type.",
    )
    parser.add_argument(
        "--no-links",
        action="store_true",
        help="Do not draw central links for inter-chromosomal TRA/BND events.",
    )
    parser.add_argument(
        "--max-links-per-figure",
        type=int,
        default=MAX_LINKS_PER_FIGURE,
        help="Maximum number of central TRA/BND links per figure. Links are evenly downsampled if needed.",
    )
    return parser.parse_args()


def import_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.path import Path as MplPath
        from matplotlib.patches import PathPatch, Wedge, Patch
        return plt, MplPath, PathPatch, Wedge, Patch
    except Exception as exc:
        raise SystemExit(
            "Cannot import matplotlib. Install it in the active conda environment, for example:\n"
            "conda install -c conda-forge matplotlib\n"
            f"Original error: {exc}"
        )


def normalize_key(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def choose_column(fieldnames, candidates, required=True):
    norm_to_real = {normalize_key(x): x for x in fieldnames}
    for candidate in candidates:
        key = normalize_key(candidate)
        if key in norm_to_real:
            return norm_to_real[key]
    if required:
        raise SystemExit(
            "Cannot find required column. Tried: "
            + ", ".join(candidates)
            + "\nAvailable columns: "
            + ", ".join(fieldnames)
        )
    return None


def to_int(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s in {"", ".", "NA", "nan", "None"}:
            return None
        return int(float(s))
    except Exception:
        return None


def norm_tool(value):
    return str(value or "").strip().lower()


def norm_chr(value):
    s = str(value or "").strip()
    if not s:
        return ""
    s = re.sub(r"^chr", "", s, flags=re.IGNORECASE)
    s = s.upper()
    if s == "23":
        return "X"
    if s == "24":
        return "Y"
    if s in {"M", "MT"}:
        return "MT"
    return s


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DEL", "DELETION"}:
        return "DEL"
    if sv in {"DUP", "DUPLICATION", "IDUP", "DUP:TANDEM", "TANDEM_DUPLICATION"}:
        return "DUP"
    if sv in {"INV", "INVERSION"}:
        return "INV"
    if sv in {"INS", "INSERTION"}:
        return "INS"
    if sv in {"TRA", "TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BND", "BREAKEND"}:
        return "BND"
    if sv in {"SGL", "SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def read_pair_order(manifest):
    path = Path(manifest)
    if not path.exists():
        return []

    pairs = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            pair_id = row.get("pair_id")
            if not pair_id:
                tumor = row.get("tumor_id", "")
                normal = row.get("normal_id", "")
                pair_id = f"{tumor}_vs_{normal}" if tumor and normal else ""
            if pair_id:
                pairs.append(pair_id)
    return pairs


def read_records(records_path):
    # records[pair_id] = list of normalized records from all six tools.
    records = defaultdict(list)
    pair_seen = OrderedDict()
    skipped = defaultdict(int)

    with open(records_path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        fieldnames = reader.fieldnames or []
        cols = {
            "pair": choose_column(fieldnames, ["pair_id", "pair", "pairid"]),
            "tool": choose_column(fieldnames, ["tool", "caller"]),
            "svtype": choose_column(fieldnames, ["svtype", "sv_type", "type"]),
            "chr1": choose_column(fieldnames, ["chrom1", "chr1", "chrom", "chr"], required=False),
            "pos1": choose_column(fieldnames, ["pos1", "start1", "pos", "start"], required=False),
            "chr2": choose_column(fieldnames, ["chrom2", "chr2", "chrom_b", "chr_b"], required=False),
            "pos2": choose_column(fieldnames, ["pos2", "end2", "end", "stop"], required=False),
        }
        if not cols["chr1"] or not cols["pos1"]:
            raise SystemExit("Need chromosome and position columns such as chrom1/pos1 or chr/start.")

        for row in reader:
            pair_id = row.get(cols["pair"], "").strip()
            tool = norm_tool(row.get(cols["tool"]))
            svtype = norm_svtype(row.get(cols["svtype"]))
            chr1 = norm_chr(row.get(cols["chr1"]))
            chr2 = norm_chr(row.get(cols["chr2"])) if cols["chr2"] else chr1
            pos1 = to_int(row.get(cols["pos1"]))
            pos2 = to_int(row.get(cols["pos2"])) if cols["pos2"] else pos1

            if not pair_id or tool not in TOOLS:
                skipped["bad_pair_or_tool"] += 1
                continue
            if chr1 not in CHR_LENGTHS or pos1 is None:
                skipped["bad_primary_endpoint"] += 1
                continue

            pair_seen[pair_id] = True
            if chr2 not in CHR_LENGTHS:
                chr2 = chr1
                pos2 = pos1
            if pos2 is None:
                pos2 = pos1

            pos1 = max(1, min(pos1, CHR_LENGTHS[chr1]))
            pos2 = max(1, min(pos2, CHR_LENGTHS[chr2]))
            records[pair_id].append(
                {
                    "pair_id": pair_id,
                    "tool": tool,
                    "svtype": svtype,
                    "chr1": chr1,
                    "pos1": pos1,
                    "chr2": chr2,
                    "pos2": pos2,
                }
            )

    return records, list(pair_seen.keys()), skipped


def build_chromosome_angles():
    total_len = sum(CHR_LENGTHS.values())
    usable_rad = 2 * math.pi - CHR_GAP_RAD * len(CHR_ORDER)
    angles = {}
    theta = math.pi / 2
    for chrom in CHR_ORDER:
        span = usable_rad * CHR_LENGTHS[chrom] / total_len
        start = theta
        end = theta - span
        angles[chrom] = (start, end)
        theta = end - CHR_GAP_RAD
    return angles


def angle_for(chrom, pos, angles):
    start, end = angles[chrom]
    frac = max(0.0, min(1.0, pos / CHR_LENGTHS[chrom]))
    return start + (end - start) * frac


def polar_point(radius, theta):
    return radius * math.cos(theta), radius * math.sin(theta)


def svtype_rings(records, show_empty=False):
    present = {rec["svtype"] for rec in records}
    if show_empty:
        svtypes = [sv for sv in SVTYPE_ORDER]
        for sv in sorted(present - set(SVTYPE_ORDER)):
            svtypes.append(sv)
        return svtypes
    priority = {sv: i for i, sv in enumerate(SVTYPE_ORDER)}
    return sorted(present, key=lambda sv: (priority.get(sv, len(SVTYPE_ORDER)), sv))


def svtype_radius_map(svtypes):
    # Kept for backward compatibility with older helper names.
    return svtype_band_map(svtypes)


def svtype_band_map(svtypes):
    if not svtypes:
        return {}, SV_BAND_TARGET_INNER

    n = len(svtypes)
    available = SV_BAND_OUTER - SV_BAND_TARGET_INNER - SV_BAND_GAP * max(0, n - 1)
    width = available / n if n else SV_BAND_WIDTH_MAX
    width = max(SV_BAND_WIDTH_MIN, min(SV_BAND_WIDTH_MAX, width))

    bands = {}
    outer = SV_BAND_OUTER
    for sv in svtypes:
        inner = outer - width
        bands[sv] = {
            "outer": outer,
            "inner": inner,
            "center": (outer + inner) / 2,
            "width": width,
        }
        outer = inner - SV_BAND_GAP

    innermost_inner = min(b["inner"] for b in bands.values())
    return bands, innermost_inner


def dynamic_link_radius(innermost_inner):
    # Keep central links inside the innermost SVTYPE band, with a fixed margin.
    # This prevents TRA/BND curves from colliding with inner SV tracks when many SVTYPE bands are present.
    return max(LINK_RADIUS_MIN, min(LINK_RADIUS_DEFAULT, innermost_inner - LINK_TRACK_MARGIN))


def tool_radius_in_band(band, tool):
    idx = TOOLS.index(tool)
    usable = max(0.001, band["width"] - 2 * TOOL_TRACK_PADDING)
    track_width = usable / len(TOOLS)
    return band["inner"] + TOOL_TRACK_PADDING + (idx + 0.5) * track_width


def tick_half_length_for_band(band):
    usable = max(0.001, band["width"] - 2 * TOOL_TRACK_PADDING)
    track_width = usable / len(TOOLS)
    return min(TICK_LENGTH / 2, track_width * 0.42)


def draw_chromosome_ring(ax, Wedge, angles):
    for i, chrom in enumerate(CHR_ORDER):
        start, end = angles[chrom]
        start_deg = math.degrees(start)
        end_deg = math.degrees(end)
        face = "#F5F7FA" if i % 2 == 0 else "#E8EDF2"
        ax.add_patch(
            Wedge(
                (0, 0),
                OUTER_RADIUS,
                math.degrees(end),
                math.degrees(start),
                width=CHR_RING_WIDTH,
                facecolor=face,
                edgecolor="#2F3A45",
                linewidth=0.75,
            )
        )

        mid = (start + end) / 2
        x, y = polar_point(CHR_LABEL_RADIUS, mid)
        rot = math.degrees(mid)
        if rot < -90 or rot > 90:
            text_rot = rot + 180
            ha = "right"
        else:
            text_rot = rot
            ha = "left"
        ax.text(
            x,
            y,
            chrom,
            fontsize=CHR_LABEL_FONTSIZE,
            rotation=text_rot,
            rotation_mode="anchor",
            ha=ha,
            va="center",
            color="#111111",
        )


def draw_chromosome_sectors(ax, angles):
    if not DRAW_CHR_SECTORS:
        return
    boundaries = []
    for chrom in CHR_ORDER:
        start, end = angles[chrom]
        boundaries.extend([start, end])

    # Deduplicate nearly identical boundaries caused by adjacent chromosome gaps.
    unique_boundaries = []
    seen = set()
    for theta in boundaries:
        key = round(theta, 4)
        if key in seen:
            continue
        seen.add(key)
        unique_boundaries.append(theta)

    for theta in unique_boundaries:
        x1, y1 = polar_point(SECTOR_INNER_RADIUS, theta)
        x2, y2 = polar_point(SECTOR_OUTER_RADIUS, theta)
        ax.plot(
            [x1, x2],
            [y1, y2],
            color=SECTOR_COLOR,
            lw=SECTOR_LINE_WIDTH,
            alpha=SECTOR_ALPHA,
            zorder=1,
        )


def draw_svtype_guides(ax, Wedge, svtypes, band_by_sv):

    for sv in svtypes:
        band = band_by_sv[sv]
        ax.add_patch(
            Wedge(
                (0, 0),
                band["outer"],
                0,
                360,
                width=band["width"],
                facecolor=SVTYPE_BAND_COLORS.get(sv, "#F7F7F7"),
                edgecolor=SV_BAND_EDGE_COLOR,
                linewidth=SV_BAND_EDGE_WIDTH,
                alpha=SV_BAND_ALPHA,
                zorder=0,
            )
        )


    legend_cx = -1.40
    legend_cy = 0.0
    theta1 = 85     
    theta2 = 95

    for sv in svtypes:
        band = band_by_sv[sv]
        ax.add_patch(
            Wedge(
                (legend_cx, legend_cy),
                band["outer"],
                theta1,
                theta2,
                width=band["width"],
                facecolor=SVTYPE_BAND_COLORS.get(sv, "#F7F7F7"),
                edgecolor="#444444",  
                linewidth=0.6,
                alpha=0.95,
                zorder=5,
            )
        )

 
        mid_theta = math.radians((theta1 + theta2) / 2)
        label_r = (band["outer"] + band["inner"]) / 2
        lx = legend_cx + label_r * math.cos(mid_theta) + 0.08
        ly = legend_cy + label_r * math.sin(mid_theta)
        ax.text(
            lx,
            ly,
            sv,
            fontsize=SVTYPE_LABEL_FONTSIZE,
            ha="left",
            va="center",
            color="#333333",
            zorder=6,
        )


def plt_circle(center, radius, **kwargs):
    import matplotlib.patches as patches
    return patches.Circle(center, radius, **kwargs)


def draw_sv_tick(ax, angles, rec, band_by_sv):
    sv = rec["svtype"]
    if sv not in band_by_sv:
        return
    band = band_by_sv[sv]
    base_r = tool_radius_in_band(band, rec["tool"])
    tick_half = tick_half_length_for_band(band)
    color = TOOL_COLORS[rec["tool"]]

    theta = angle_for(rec["chr1"], rec["pos1"], angles)
    r1 = base_r - tick_half
    r2 = base_r + tick_half
    x1, y1 = polar_point(r1, theta)
    x2, y2 = polar_point(r2, theta)
    ax.plot(
        [x1, x2],
        [y1, y2],
        color=color,
        alpha=TICK_ALPHA,
        lw=TICK_LINE_WIDTH,
        solid_capstyle="round",
        zorder=4,
    )

    # Also mark the second chromosome endpoint for inter-chromosomal events.
    if rec["chr2"] != rec["chr1"]:
        theta2 = angle_for(rec["chr2"], rec["pos2"], angles)
        a1, b1 = polar_point(r1, theta2)
        a2, b2 = polar_point(r2, theta2)
        ax.plot(
            [a1, a2],
            [b1, b2],
            color=color,
            alpha=TICK_ALPHA,
            lw=TICK_LINE_WIDTH,
            solid_capstyle="round",
            zorder=4,
        )


def evenly_downsample(items, max_n):
    if len(items) <= max_n:
        return items
    if max_n <= 0:
        return []
    step = len(items) / max_n
    return [items[int(i * step)] for i in range(max_n)]


def draw_trans_link(ax, MplPath, PathPatch, angles, rec, link_radius):
    theta1 = angle_for(rec["chr1"], rec["pos1"], angles)
    theta2 = angle_for(rec["chr2"], rec["pos2"], angles)
    p0 = polar_point(link_radius, theta1)
    p3 = polar_point(link_radius, theta2)
    p1 = (p0[0] * 0.16, p0[1] * 0.16)
    p2 = (p3[0] * 0.16, p3[1] * 0.16)
    path = MplPath(
        [p0, p1, p2, p3],
        [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4],
    )
    ax.add_patch(
        PathPatch(
            path,
            facecolor="none",
            edgecolor=TOOL_COLORS[rec["tool"]],
            lw=LINK_WIDTH,
            alpha=LINK_ALPHA,
            capstyle="round",
            joinstyle="round",
            zorder=3,
        )
    )


def count_by_tool(records):
    counts = defaultdict(int)
    for rec in records:
        counts[rec["tool"]] += 1
    return counts


def count_by_svtype(records):
    counts = defaultdict(int)
    for rec in records:
        counts[rec["svtype"]] += 1
    return counts


def write_pair_matrix(outdir, pair_id, records):
    safe_pair = re.sub(r"[^A-Za-z0-9_.-]+", "_", pair_id)
    path = Path(outdir) / "tables" / f"{safe_pair}.svtype_tool_counts.tsv"
    path.parent.mkdir(parents=True, exist_ok=True)
    counter = defaultdict(int)
    for rec in records:
        counter[(rec["svtype"], rec["tool"])] += 1

    svtypes = svtype_rings(records, show_empty=True)
    with open(path, "wt") as handle:
        handle.write("svtype\t" + "\t".join(TOOLS) + "\ttotal\n")
        for sv in svtypes:
            vals = [counter.get((sv, tool), 0) for tool in TOOLS]
            if sum(vals) == 0:
                continue
            handle.write(sv + "\t" + "\t".join(str(v) for v in vals) + "\t" + str(sum(vals)) + "\n")
    return path


def plot_pair(plt, MplPath, PathPatch, Wedge, Patch, outdir, pair_id, records, output_format, show_empty, draw_links, max_links):
    angles = build_chromosome_angles()
    svtypes = svtype_rings(records, show_empty=show_empty)
    band_by_sv, innermost_inner = svtype_band_map(svtypes)
    link_radius = dynamic_link_radius(innermost_inner)

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), facecolor="white")
    ax.set_aspect("equal")
    ax.axis("off")

    draw_chromosome_ring(ax, Wedge, angles)
    draw_chromosome_sectors(ax, angles)
    draw_svtype_guides(ax, Wedge, svtypes, band_by_sv)

    for rec in records:
        draw_sv_tick(ax, angles, rec, band_by_sv)

    link_records = [
        rec for rec in records
        if rec["chr1"] != rec["chr2"] and rec["svtype"] in {"TRA", "BND"}
    ]
    if draw_links and link_records:
        for rec in evenly_downsample(link_records, max_links):
            draw_trans_link(ax, MplPath, PathPatch, angles, rec, link_radius)


    tool_handles = [
        Patch(facecolor=TOOL_COLORS[tool], edgecolor="none", label=TOOL_LABELS[tool])
        for tool in TOOLS
    ]
    ax.legend(
        handles=tool_handles,
        title="Tool",
        frameon=False,
        fontsize=LEGEND_FONTSIZE,
        title_fontsize=LEGEND_FONTSIZE,
        loc="center left",
        bbox_to_anchor=(1.02, 0.55),
    )

    by_tool = count_by_tool(records)
    by_sv = count_by_svtype(records)
    total = len(records)
    trans_n = len(link_records)
    title = f"{pair_id}: six-tool SVTYPE chromosome distribution"
    subtitle = (
        f"SV records={total}; TRA/BND inter-chromosomal links={trans_n}; "
        + "; ".join(f"{TOOL_LABELS[t]}={by_tool.get(t, 0)}" for t in TOOLS)
    )
    ax.text(0, 1.29, title, ha="center", va="center", fontsize=TITLE_FONTSIZE)
    ax.text(0, 1.21, subtitle, ha="center", va="center", fontsize=SUBTITLE_FONTSIZE, color="#444444")
    ax.text(
        -1.22,
        -1.23,
        "Annular bands = SVTYPE; short ticks and central links are colored by SV caller",
        ha="left",
        va="center",
        fontsize=SUBTITLE_FONTSIZE,
        color="#555555",
    )

    ax.set_xlim(-2.35, 1.55)
    ax.set_ylim(-1.56, 1.56)

    safe_pair = re.sub(r"[^A-Za-z0-9_.-]+", "_", pair_id)
    prefix = Path(outdir) / f"{safe_pair}.six_tool_svtype_circos"
    out_paths = []
    fig.canvas.draw()
    if output_format in {"png", "both"}:
        png = Path(str(prefix) + ".png")
        fig.savefig(png, dpi=DPI, bbox_inches="tight", facecolor="white", transparent=False)
        out_paths.append(png)
    if output_format in {"pdf", "both"}:
        pdf = Path(str(prefix) + ".pdf")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white", transparent=False)
        out_paths.append(pdf)
    plt.close(fig)
    return out_paths, total, trans_n, by_sv


def write_skip_summary(outdir, skipped):
    path = Path(outdir) / "pair_circos.skipped_records.summary.tsv"
    with open(path, "wt") as handle:
        handle.write("reason\tcount\n")
        for reason in sorted(skipped):
            handle.write(f"{reason}\t{skipped[reason]}\n")
    return path


def main():
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    plt, MplPath, PathPatch, Wedge, Patch = import_matplotlib()
    records_by_pair, table_pair_order, skipped = read_records(args.records)
    manifest_pair_order = read_pair_order(args.manifest)

    pair_set = set(table_pair_order)
    pair_order = [p for p in manifest_pair_order if p in pair_set] or table_pair_order

    status_path = outdir / "pair_circos_svtype_tool_rings_v3.plot_status.tsv"
    skip_path = write_skip_summary(outdir, skipped)

    fig_n = 0
    skip_no_sv = 0
    with open(status_path, "wt") as status:
        status.write("pair_id\tstatus\tsv_records\ttra_bnd_links\tmatrix\tfigures\n")
        for pair_id in pair_order:
            records = records_by_pair.get(pair_id, [])
            if not records:
                skip_no_sv += 1
                status.write(f"{pair_id}\tSKIP_NO_SV\t0\t0\tNA\tNA\n")
                continue
            matrix_path = write_pair_matrix(outdir, pair_id, records)
            fig_paths, total, trans_n, _ = plot_pair(
                plt=plt,
                MplPath=MplPath,
                PathPatch=PathPatch,
                Wedge=Wedge,
                Patch=Patch,
                outdir=outdir,
                pair_id=pair_id,
                records=records,
                output_format=args.format,
                show_empty=args.show_empty_rings,
                draw_links=(DRAW_TRA_BND_LINKS and not args.no_links),
                max_links=args.max_links_per_figure,
            )
            fig_n += len(fig_paths)
            status.write(
                f"{pair_id}\tPASS\t{total}\t{trans_n}\t{matrix_path}\t"
                f"{','.join(str(p) for p in fig_paths)}\n"
            )

    print(f"[DONE] figures={fig_n}")
    print(f"[SKIP_NO_SV] pairs={skip_no_sv}")
    print(f"[STATUS] {status_path}")
    print(f"[SKIP_TABLE] {skip_path}")
    print(f"[OUTDIR] {outdir}")


if __name__ == "__main__":
    main()


### 运行：

In [ ]:
python plot_pair_circos_svtype_tool_rings.py \
  --records /mnt/home/ygjx/chenkejin/80_sv_summary/tables/all_pass_sv_records.normalized.tsv \
  --manifest /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv \
  --outdir /mnt/home/ygjx/chenkejin/80_sv_summary/plots/pair_circos_svtype_tool_rings \
  --format both